# Bloque 2 · De datos dispersos a estadísticas fiables — prácticas en Google Colab

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/consumer-spain/curso-ia-colab/blob/main/bloque2/bloque2_practicas.ipynb)

**Esto es el Plan B.** Si en tu ordenador algo no funciona (Python, paquetes, permisos…), puedes seguir aquí
**todas las prácticas del Bloque 2**, en el mismo orden que en clase.

**Cómo se usa**
- Cada bloque gris es una *celda*. Se ejecuta con el botón ▶ de su izquierda (o **Mayús + Enter**).
- Ejecuta las celdas **en orden**, de arriba abajo.
- Si Colab se desconecta o lo reinicias, vuelve a ejecutar la celda **Preparación**.
- Colab **no guarda tus cambios**. Si quieres conservarlos: *Archivo → Guardar una copia en Drive*.
- No subas aquí datos reales ni información interna: todo el material del curso es de prueba (sintético).
- La IA que uses para generar el código es la misma que en clase: este cuaderno **no** la sustituye. Tú le pides el
  código, lo pegas en la celda indicada y lo ejecutas.

| En clase | Aquí |
|---|---|
| Doble clic en un `.bat`, VS Code o la terminal de Windows | Ejecutar una celda |
| `py -3 archivo.py` / `python archivo.py` | `!python archivo.py` |
| Guardar el script que te da la IA (VS Code, Bloc de notas) | Celda `%%writefile archivo.py` y pegar el código |
| Abrir un Excel con Excel | `ver_excel("ruta.xlsx")` en una celda, o `descargar("ruta.xlsx")` para abrirlo en tu equipo |
| Navegar por carpetas con el Explorador | `ir_a("carpeta")` y `!ls` / `!find` |
| Adjuntar archivos a tu IA arrastrándolos | `descargar(...)` y subirlos al chat de tu IA |
| Ventanas con botones (tkinter) | No funcionan en Colab (ver abajo) |

**Cómo se clasifica cada práctica** (verás una de estas etiquetas donde ocurre):

- 🟢 **IGUAL** — se hace exactamente igual que en clase.
- 🟡 **ADAPTADA** — se hace con otra herramienta, pero se aprende lo mismo.
- 🔴 **NO EXPORTABLE** — no se puede hacer en Colab; se dice qué hacer en su lugar.

**Cuándo no mirar la solución.** En clase las referencias están disponibles desde el principio, pero usar una
referencia forma parte del ejercicio, y comprobarla también. Están todas en la última sección,
**«Solo si te has bloqueado»**.

## Preparación

Descarga los materiales del Bloque 2 (el mismo ZIP que en clase), define unas pequeñas herramientas
(`ir_a`, `ver_excel`, `descargar`…) y comprueba que todo está listo.

> 🟡 **ADAPTADA** — en clase se instalan los requisitos con `py -3 -m pip install -r requirements.txt`; aquí lo hace esta celda.

In [ ]:
# PREPARACIÓN · ejecuta esta celda primero
import os, sys, zipfile, urllib.request, subprocess, shutil, re
from pathlib import Path

ZIP_URL = "https://raw.githubusercontent.com/consumer-spain/curso-ia-colab/main/bloque2/BLOQUE2_MATERIALES.zip"
NOMBRE_BASE = "BLOQUE2_MATERIALES_ACTUALIZADOS"
_aqui = Path.cwd().resolve()
# Si repites esta celda estando dentro de la carpeta de materiales, seguimos trabajando en la carpeta de fuera (no se anidan copias)
TRABAJO = Path(*_aqui.parts[:_aqui.parts.index(NOMBRE_BASE)]) if NOMBRE_BASE in _aqui.parts else _aqui
BASE = TRABAJO / NOMBRE_BASE
ZIP_LOCAL = TRABAJO / "materiales_b2.zip"

if not BASE.exists():
    print("Descargando los materiales del Bloque 2...")
    urllib.request.urlretrieve(ZIP_URL, ZIP_LOCAL)
    with zipfile.ZipFile(ZIP_LOCAL) as z:
        z.extractall(TRABAJO)
    print("Materiales listos en:", BASE)
else:
    print("Los materiales ya estaban descargados:", BASE)

# Paquetes (normalmente Colab ya los trae)
for paquete in ("pandas", "openpyxl"):
    try:
        __import__(paquete)
    except ImportError:
        print("Instalando", paquete, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)


def ir_a(carpeta):
    """Se coloca en una carpeta del material (como abrir la carpeta en el Explorador)."""
    os.chdir(BASE / carpeta if carpeta else BASE)
    print("Estás en:", Path.cwd().relative_to(TRABAJO))


def hojas(ruta):
    """Lista las hojas de un libro de Excel."""
    from openpyxl import load_workbook
    if not Path(ruta).exists():
        print("⚠ Todavía no existe:", Path(ruta).name, "· ¿has ejecutado antes el programa que lo crea?")
        return
    wb = load_workbook(ruta, data_only=True)
    print(Path(ruta).name)
    for ws in wb.worksheets:
        print("  ·", ws.title, "·", ws.dimensions)


def ver_excel(ruta, hoja=None, filas=(1, 15), columnas=8, buscar=None):
    """Muestra una hoja de Excel como cuadrícula, con números de fila y letras de columna (como en Excel).
    filas=(desde, hasta) · columnas=cuántas columnas · buscar="texto": solo la fila 1 y las filas que lo contienen."""
    import pandas as pd
    from openpyxl import load_workbook
    from openpyxl.utils import get_column_letter
    from IPython.display import display
    ruta = Path(ruta)
    if not ruta.exists():
        print("⚠ Todavía no existe:", ruta.name, "· ¿has ejecutado antes el programa que lo crea?")
        return
    wb = load_workbook(ruta, data_only=True)
    if hoja is None:
        hoja = wb.sheetnames[0]
    if hoja not in wb.sheetnames:
        print("⚠ No hay ninguna hoja", repr(hoja), "· hojas disponibles:", wb.sheetnames)
        return
    ws = wb[hoja]
    ncol = min(columnas, ws.max_column)
    letras = [get_column_letter(i) for i in range(1, ncol + 1)]
    datos = {}
    if buscar:
        for r in range(1, ws.max_row + 1):
            vals = [ws.cell(r, c).value for c in range(1, ncol + 1)]
            if r == 1 or any(isinstance(v, str) and buscar.lower() in v.lower() for v in vals):
                datos[r] = vals
    else:
        for r in range(filas[0], min(filas[1], ws.max_row) + 1):
            datos[r] = [ws.cell(r, c).value for c in range(1, ncol + 1)]
    df = pd.DataFrame.from_dict({r: ["" if v is None else v for v in vs] for r, vs in datos.items()},
                                orient="index", columns=letras)
    df.index.name = "fila"
    print(f"{ruta.name} · hoja «{hoja}» · {ws.dimensions}")
    with pd.option_context("display.max_colwidth", 45, "display.width", 250, "display.max_columns", 30):
        display(df)


def graficos(ruta):
    """Lista las gráficas de un libro de Excel (pandas no las ve; para verlas, descarga el archivo)."""
    if not Path(ruta).exists():
        print("⚠ Todavía no existe:", Path(ruta).name, "· ¿has ejecutado antes el programa que lo crea?")
        return
    with zipfile.ZipFile(ruta) as z:
        nombres = sorted(n for n in z.namelist() if re.search(r"charts/chart\d+\.xml$", n))
        for n in nombres:
            xml = z.read(n).decode("utf8")
            m = re.search(r"<(?:\w+:)?title>.*?</(?:\w+:)?title>", xml, re.S)
            titulo = " ".join(re.findall(r"<(?:\w+:)?t>(.*?)</(?:\w+:)?t>", m.group(0))) if m else "(sin título)"
            print(" ·", titulo)
    print(len(nombres), "gráficas en", Path(ruta).name)


def descargar(ruta):
    """Descarga un archivo a tu ordenador (para abrirlo con Excel o subirlo a tu IA)."""
    ruta = Path(ruta)
    try:
        from google.colab import files
    except ImportError:
        print("(Fuera de Colab) El archivo está en:", ruta)
        return
    files.download(str(ruta))


def descargar_carpeta(ruta):
    """Comprime una carpeta en un ZIP y lo descarga."""
    ruta = Path(ruta)
    zip_ = shutil.make_archive(str(TRABAJO / ruta.name), "zip", ruta)
    descargar(zip_)


def revelar(texto):
    """Muestra un texto con formato (para las 'revelaciones': ejecútala cuando hayas pensado tu respuesta)."""
    from IPython.display import display, Markdown
    display(Markdown(texto))


# --- Comprobación (este bloque no trae comprobador propio: esta parte hace de comprobador) ---
print()
print("COMPROBACIÓN")
necesarios = [
    "LEEME_BLOQUE2.md",
    "01_reto_documentacion/MATERIAL/listado_inspeccion.xlsx",
    "01_reto_documentacion/AYUDAS_Y_REFERENCIAS/solucion_nivel1.py",
    "02_muestra_sintetica/00_MADRID_SIMULADO_PARA_MUESTRA.xlsx",
    "03_modelo_2_ccaa/ENTRADAS_LIMPIAS/MADRID.xlsx",
    "03_modelo_2_ccaa/CONTROLES/RESULTADO_ESPERADO_2_CCAA.xlsx",
    "03_modelo_2_ccaa/01_MODELO_REFORZADO_2_CCAA.xlsx",
    "04_construccion_guiada/01_CONSOLIDAR_BASE.py",
    "05_reto_gimnasios/02_BASE_12_CCAA_INTEGRADAS.xlsx",
    "05_reto_gimnasios/CCAA_PENDIENTES/ARAGON.xlsx",
    "05_reto_gimnasios/NORMALIZACION/normalizar_entradas.py",
    "06_diagnostico_y_validacion/auditar_salida.py",
    "99_ayudas_y_referencias/02_SOLUCION_COMPLETA.py",
]
fallos = []
for rel in necesarios:
    existe = (BASE / rel).exists()
    print("  " + ("✔ " if existe else "✘ ") + rel)
    if not existe:
        fallos.append(rel)
try:
    from openpyxl import load_workbook
    load_workbook(BASE / "05_reto_gimnasios/02_BASE_12_CCAA_INTEGRADAS.xlsx", data_only=True)
    print("  ✔ Python", sys.version.split()[0], "· pandas y openpyxl funcionan · Excel legible")
except Exception as e:
    print("  ✘ No puedo leer los Excel:", e)
    fallos.append("lectura de Excel")

ir_a("")
print()
print("TODO LISTO" if not fallos else "HAY ALGO QUE REVISAR")

---
# Reconciliar dos fuentes

*(Diapositivas 4–11)*

Una unidad de vigilancia de mercado necesita saber si la documentación de un expediente está completa.
Tienes **un Excel** con los archivos que *deberían* existir y **una carpeta real** con los que existen de verdad
(con subcarpetas). La herramienta debe generar un Excel **nuevo**: nunca toca los originales.

Primero miramos el Excel y la carpeta, **sin ejecutar ningún programa todavía**.

> 🟡 **ADAPTADA** — en clase abres el Excel y la carpeta con Excel y el Explorador; aquí, con `ver_excel` y `!find`.

In [ ]:
ir_a("01_reto_documentacion")
ver_excel("MATERIAL/listado_inspeccion.xlsx", filas=(1, 26), columnas=5)

In [ ]:
# La carpeta real: fíjate en que no es plana (hay una subcarpeta)
!find MATERIAL/imagenes | sort

## Antes de automatizar: ¿cómo sabremos que está bien?

Tenemos un Excel que dice qué archivos deberían existir y una carpeta con lo que tenemos realmente.

**¿Qué tendría que ser cierto al final para poder decir que la herramienta funciona bien?**

Piénsalo (o escríbelo) **antes de seguir**. Las pistas están en la celda siguiente: no las leas todavía.

> 🟢 **IGUAL** — actividad de pensar, como en clase.

In [ ]:
#@title Ver las pistas (solo después de pensar tu respuesta)
revelar("""
**Pistas**

- ¿Qué puede pasar con cada fila del Excel?
- ¿Qué hacemos con archivos que están en la carpeta pero no en el Excel?
- ¿Qué cosas no debería modificar el programa?
""")

## Nuestros criterios de aceptación

Ejecuta la celda cuando hayas pensado tu respuesta y contrástala.

In [ ]:
#@title Ver la respuesta (ejecútala cuando hayas pensado la tuya)
revelar("""
**Para cada fila del Excel**

- **Encontrado** — el archivo existe
- **No encontrado** — debería existir, pero no está
- **Sin nombre indicado** — no se ha especificado archivo

**Además**

- ✓ buscar también dentro de subcarpetas
- ✓ detectar archivos presentes pero no registrados
- ✓ crear una hoja *Sin registrar*
- ✓ generar un Excel nuevo
- ✓ preservar el Excel y las imágenes originales

*Los criterios de comprobación se definen antes de ver el resultado.*
""")

## Reto · Comprobar la documentación de un expediente · 20 min

**Objetivo:** conseguir una solución funcional que cumpla los criterios de aceptación.

Material: `01_reto_documentacion/MATERIAL/` (`listado_inspeccion.xlsx` e `imagenes/`).
La herramienta debe generar un Excel nuevo **sin modificar los originales**.
Prompt y referencias disponibles desde el principio (están en `AYUDAS_Y_REFERENCIAS/`; ver «Solo si te has bloqueado»).

**Pásale esto a tu IA** (puedes usar el breve o el completo; son los de `PROMPTS_RETO_DOCUMENTACION.txt`, con las rutas de Colab):

```
Tengo un Excel con una hoja llamada Documentacion y una columna nombre_archivo.
También tengo una carpeta con imágenes y puede haber subcarpetas.

Quiero un script de Python que:
- compruebe para cada fila si el archivo existe;
- genere un Excel nuevo, sin modificar el original;
- añada las columnas estado y ruta_encontrada;
- utilice los estados Encontrado, No encontrado y Sin nombre indicado;
- añada una hoja Sin registrar con los archivos de la carpeta que no aparecen en el Excel;
- busque dentro de subcarpetas;
- y no mueva, borre ni modifique los archivos originales.

Explícame primero qué va a hacer y después dame:
1. el código completo;
2. las librerías necesarias;
3. cómo ejecutarlo.

Trabajo en Google Colab, no en Windows: lo ejecutaré con "!python mi_comprobador.py" desde la carpeta
01_reto_documentacion. El Excel está en MATERIAL/listado_inspeccion.xlsx, las imágenes en MATERIAL/imagenes
y el resultado debe guardarse en SALIDAS/mi_resultado.xlsx.
```

Cuando tengas el código: **pégalo en la celda de abajo** (borrando las dos líneas que hay) y ejecútala.
El nombre del archivo ya está puesto.

> 🟡 **ADAPTADA** — en clase guardas el script con VS Code y lo ejecutas con `py -3`; aquí lo guardas con `%%writefile` y lo ejecutas con `!python`. Si tu IA te da instrucciones para Windows (`py -3`), ignóralas.

In [ ]:
ir_a("01_reto_documentacion")

In [ ]:
%%writefile mi_comprobador.py
# ↓ Borra estas dos líneas y pega aquí, completo, el código que te ha dado tu IA ↓

In [ ]:
# Ejecutar tu programa (con !python, para ver el traceback completo si algo falla)
!python mi_comprobador.py

> 🔴 **NO EXPORTABLE** — **Ampliación opcional · interfaz** (selector de Excel y carpeta, ventanas con resumen y errores, con `tkinter`):
> en Colab no se abren ventanas, así que no se puede hacer ni ver aquí. No es necesaria para completar el reto.
> **En su lugar**, si terminas pronto: pídele a tu IA que tu programa reciba por línea de comandos el Excel, la carpeta y
> el archivo de salida (`--excel`, `--carpeta`, `--salida`) y que imprima un resumen final. Es la misma idea sin ventanas.

### Checkpoint · minuto 10 — ¿En qué punto estás?

1. ¿Tienes una primera salida?
2. ¿Busca en subcarpetas?
3. ¿Has abierto el Excel generado?

Si estás bloqueado, utiliza una ayuda o referencia.

La celda siguiente abre **tu** Excel generado (equivale a abrirlo con Excel):

> 🟡 **ADAPTADA** — abrir el Excel con Excel → `ver_excel`.

In [ ]:
ver_excel("SALIDAS/mi_resultado.xlsx", filas=(1, 28), columnas=7)

In [ ]:
# Las hojas que ha creado tu programa
hojas("SALIDAS/mi_resultado.xlsx")

## Validación colectiva: ¿ha hecho realmente lo que pedíamos?

En clase, el grupo pone en común cuántos *Encontrado / No encontrado / Sin nombre indicado / Sin registrar* ha dado cada herramienta.
Aquí: **cuenta los tuyos** (tu programa debería haber puesto un estado por fila y una hoja `Sin registrar`).

> 🟡 **ADAPTADA** — los cuentas con pandas en lugar de mirar el Excel; la comparación con el resultado de referencia es igual que en clase.

In [ ]:
import pandas as pd
try:
    df = pd.read_excel("SALIDAS/mi_resultado.xlsx", sheet_name=0)
    print(df["estado"].value_counts())
except Exception as e:
    print("No he podido leer tu resultado todavía:", e)

In [ ]:
# ¿Siguen intactos los originales? Mira las fechas de modificación: deben ser anteriores a tu ejecución.
!ls -l --time-style=long-iso MATERIAL MATERIAL/imagenes | head -12

Cuando tengas tus números, ejecuta la celda siguiente para ver el **resultado de referencia** y compáralo.

In [ ]:
#@title Ver la respuesta (ejecútala cuando hayas pensado la tuya)
revelar("""
**Resultado de referencia**

| Encontrados | No encontrados | Sin nombre indicado | Sin registrar |
|:-:|:-:|:-:|:-:|
| **21** | **3** | **1** | **2** |

- ✓ `CAPTURA_007.jpg` se encuentra dentro de una subcarpeta
- ✓ se crea la hoja *Sin registrar*
- ✓ los originales permanecen intactos

*No hemos comparado tu código con el mío: hemos comprobado si el resultado cumple los criterios que definimos antes.*
""")

## Una herramienta fiable empieza antes de programar

1. Definimos primero qué significaba «correcto».
2. Automatizamos la comparación.
3. Validamos el resultado con criterios independientes del código.

---
# Una campaña estatal genera un problema de datos

*(Diapositivas 12–15)*

Varias autoridades autonómicas realizan una misma campaña → la información llega descentralizada → necesitamos
**una visión estatal reproducible**: un único resultado estatal, que podamos regenerar cuando cambien los datos.

*Caso inspirado en un problema real · todos los datos del curso son sintéticos.*

## ¿Basta con este encargo?

Ya sabemos en qué consiste la campaña y que queremos una visión estatal reproducible.

> *«Hazme un programa en Python que junte estos Excel, saque estadísticas y gráficas.»*

**¿Basta con esto? ¿Qué información necesitaría la IA antes de escribir código?**
Apunta 4–6 respuestas: las recuperaremos al final.

> 🟢 **IGUAL** — actividad de pensar.

## Las 8 piezas del encargo

Ordena ahora lo que has apuntado. No es una plantilla universal ni un «prompt mágico»: es un mapa de las decisiones
que hay que tomar **para este problema**.

1. **Objetivo** — ¿para qué quiero la herramienta?
2. **Estructura** — ¿qué significan mis datos?
3. **Restricciones** — ¿qué puedo hacer o compartir?
4. **Reglas** — ¿cómo debe calcularse?
5. **Controles** — ¿cómo demuestro que funciona?
6. **Resultado** — ¿qué debe entregar exactamente?
7. **Archivos** — ¿qué fuentes concretas procesa?
8. **Variaciones** — ¿qué diferencias debe tolerar?

Los iremos resolviendo uno a uno. Los mensajes a la IA se construyen **pieza a pieza**, en una misma conversación.

> 🟢 **IGUAL** — mapa de decisiones, como en clase.

---
# OBJETIVO · ¿Para qué queremos la herramienta?

*(Diapositivas 16–18)* · 0 / 8 resueltas

**Objetivo ≠ resultado.**

- **OBJETIVO = finalidad.** ¿Para qué quiero construir la herramienta? *«Quiero convertir los resultados autonómicos de una campaña en una visión estatal que pueda regenerarse cuando cambien los datos.»*
- **RESULTADO = artefactos.** ¿Qué debe entregar exactamente? Hojas, tablas, gráficas, controles… *(lo decidiremos más adelante)*.

Primero, para qué. Después, qué debe entregar.

## Primer mensaje: qué queremos conseguir

Abre una **conversación nueva** con tu IA y envía el **MENSAJE 1**. (Es el de `04_construccion_guiada/prompt_construccion_guiada.txt`.)

```
Necesito una herramienta que me permita integrar los resultados de una campaña de vigilancia de mercado realizada por distintas comunidades autónomas y generar una visión estatal que pueda actualizarse cuando cambien los datos.

Todavía no escribas código. Antes, dime qué información necesitas sobre los archivos para poder hacerlo bien.
```

*No escribimos un prompt mágico de una vez: convertimos decisiones en instrucciones.*

> 🟢 **IGUAL** — la conversación con tu IA es la misma que en clase.

---
# ESTRUCTURA · ¿Qué significan mis datos?

*(Diapositivas 19–22)* · 1 / 8 resueltas (OBJETIVO ✓)

## Una entrada limpia de referencia: Madrid

Abre `02_muestra_sintetica/00_MADRID_SIMULADO_PARA_MUESTRA.xlsx`, hoja `MADRID_DATOS`, filas 1–11.
(El libro tiene además las hojas `MADRID_ESTADISTICAS` e `INSTRUCCIONES`.)

> 🟡 **ADAPTADA** — abrir el Excel → `ver_excel`. Verás los números de fila y las letras de columna, como en Excel.

In [ ]:
ir_a("02_muestra_sintetica")
hojas("00_MADRID_SIMULADO_PARA_MUESTRA.xlsx")
ver_excel("00_MADRID_SIMULADO_PARA_MUESTRA.xlsx", "MADRID_DATOS", filas=(1, 11), columnas=4)

Fíjate en:

1. **1 fila = 1 pregunta** del formulario común de la campaña · 25 preguntas.
2. **Sí · No · No procede**: cuántos controles dieron cada respuesta.
3. **Sí + No + No procede** = controles realizados para esa pregunta.
4. **3 bloques**: preguntas 1–5 · 6–14 · 15–25.

Pregunta 1 · 12 Sí + 15 No + 0 No procede = 27 controles realizados.

## Microactividad · 1–2 min · Explica el archivo sin enseñarlo

Si tuvieras que explicar este archivo a alguien que **no puede verlo**, ¿qué información sería imprescindible contarle?
*Sin programar. Pensadlo en voz alta.*

## Segundo mensaje: cómo son los datos

Envía el **MENSAJE 2** en la misma conversación:

```
Cada archivo corresponde a una comunidad autónoma y contiene una tabla con las 25 preguntas del formulario común de la campaña.

Para cada pregunta hay tres recuentos: Sí, No y No procede. Su suma es el número de controles realizados en esa pregunta.

Las preguntas se agrupan en tres bloques: 1–5, 6–14 y 15–25.
```

*Para programar bien hay que describir el significado de los datos, no solo adjuntar un fichero.*

> 🟢 **IGUAL** — el mensaje a tu IA es el mismo que en clase.

---
# RESTRICCIONES · ¿Qué podemos hacer y compartir?

*(Diapositivas 23–31)* · 2 / 8 resueltas (OBJETIVO ✓, ESTRUCTURA ✓)

Para pedir ayuda a la IA tendremos que darle información suficiente sobre cómo son los datos. ¿Necesitamos enseñarle el archivo tal cual?
**Que tengamos el archivo no significa que podamos enviarlo tal cual a cualquier servicio.**

## Antes de compartir un archivo con una IA

Primero preguntamos qué contiene y dónde lo vamos a enviar.

1. ¿Contiene datos personales o información confidencial?
2. ¿El servicio está autorizado para tratar ese contenido?
3. ¿Necesita realmente la IA todos esos datos?

Si el entorno no está autorizado para ese contenido, no lo subimos tal cual.
*Entornos corporativos autorizados · modelos locales · otras soluciones aprobadas por el organismo · reglas internas del organismo.*

> Lo mismo vale para este cuaderno: Colab es un servicio de Google. **No subas aquí datos reales.**
> 🟢 **IGUAL** — actividad de decisión, igual que en clase.

## Reducir lo que compartimos

- **Eliminar:** quitar columnas, filas u hojas que la tarea no necesita.
- **Reducir:** compartir solo una parte representativa.
- **Sustituir:** reemplazar valores concretos por otros que no necesiten ser reales.
- **Generar datos sintéticos:** un archivo nuevo, con la misma forma y datos ficticios.

Muchas veces basta con darle a la IA la forma de los datos, no los valores reales.
*Quitar identificadores ≠ necesariamente anonimizar.*

## ¿Qué necesitamos conservar para que la muestra siga siendo útil?

Si sustituimos los datos de Madrid por datos ficticios, **¿qué debe seguir igual y qué puede cambiar?**
Piénsalo y ejecuta la celda siguiente para contrastar.

In [ ]:
#@title Ver la respuesta (ejecútala cuando hayas pensado la tuya)
revelar("""
| SE MANTIENE | CAMBIA | SE CONSERVA |
|---|---|---|
| las 25 preguntas · los encabezados · la estructura de la tabla · los tipos de dato · las relaciones internas | los recuentos reales → enteros no negativos ficticios | el total de controles por pregunta (Sí + No + No procede = mismo total) |

*Una muestra útil conserva las propiedades que necesita la tarea, no los valores reales.*
""")

## De las decisiones al script (encargo aparte)

**Problema** (no compartir los recuentos reales) + **requisitos** (lo que acabamos de decidir) → **prompt** → **script** `generar_muestra_sintetica.py`.
Es un encargo pequeño y aparte: no lo mezcles con la conversación principal de la herramienta.

Ejemplo de prompt:

```
Tengo el archivo 00_MADRID_SIMULADO_PARA_MUESTRA.xlsx, con 25 preguntas y recuentos Sí / No / No procede.
Escribe un script en Python que cree una COPIA nueva del archivo en la que:
  · se conserven las 25 preguntas y la misma estructura de columnas;
  · los recuentos se sustituyan por enteros no negativos generados al azar;
  · el total Sí + No + No procede de cada pregunta sea el mismo que en el original.
No modifiques el archivo original.
```

## Mini-práctica · Genera una muestra sintética

Pasos: **1** pedir el script → **2** revisarlo → **3** guardarlo → **4** ejecutarlo → **5** abrir el resultado.
Usa el prompt de arriba o escribe el tuyo con las mismas decisiones. Para Colab, añádele al final esta línea:

```
Lo ejecutaré en Google Colab con "!python mi_generar_muestra.py" desde la carpeta donde está el Excel. Guarda la copia nueva como MI_MUESTRA_MADRID.xlsx en esa misma carpeta.
```

> 🟡 **ADAPTADA** — guardas el script con `%%writefile` y lo ejecutas con `!python`.

In [ ]:
ir_a("02_muestra_sintetica")

In [ ]:
%%writefile mi_generar_muestra.py
# ↓ Borra estas dos líneas y pega aquí, completo, el código que te ha dado tu IA ↓

In [ ]:
!python mi_generar_muestra.py

In [ ]:
# Abre el resultado (paso 5)
hojas("MI_MUESTRA_MADRID.xlsx")
ver_excel("MI_MUESTRA_MADRID.xlsx", filas=(1, 12), columnas=4)

## Antes y después: qué significa «conservar propiedades»

Compara, pregunta a pregunta, el **ANTES** (entrada limpia de Madrid) con el **DESPUÉS** (tu muestra sintética).
¿Qué es igual y qué es distinto?

In [ ]:
print("ANTES · entrada limpia (Madrid)")
ver_excel("00_MADRID_SIMULADO_PARA_MUESTRA.xlsx", "MADRID_DATOS", filas=(5, 11), columnas=4)
print()
print("DESPUÉS · tu muestra sintética")
ver_excel("MI_MUESTRA_MADRID.xlsx", filas=(5, 11), columnas=4)

In [ ]:
#@title Ver la respuesta (ejecútala cuando hayas pensado la tuya)
revelar("""
Con la muestra de referencia (semilla 42), la **pregunta 1** pasa de `12 · 15 · 0` a `14 · 13 · 0`:

| | ANTES | DESPUÉS | |
|---|:-:|:-:|:-:|
| PREGUNTA | Pregunta 1 | Pregunta 1 | = igual |
| COLUMNAS | Sí · No · No procede | Sí · No · No procede | = iguales |
| RECUENTOS | 12 · 15 · 0 | 14 · 13 · 0 | ≠ diferentes |
| TOTAL | 27 | 27 | = igual |

Tus valores serán otros (tu script usará otra aleatoriedad): comprueba que **también suman 27**.
*Esto es lo que significa conservar propiedades: cambian los valores, se mantiene todo lo que la tarea necesita.*
""")

## Comprueba la muestra antes de utilizarla

Revisa **toda** tu muestra (no solo la pregunta 1) contra esta lista, mirando el Excel con `ver_excel`
(cambia `filas=(1, 33)` para ver las 25 preguntas):

- □ 25 preguntas
- □ misma estructura
- □ recuentos cambiados
- □ total conservado en cada pregunta
- □ archivo nuevo
- □ original intacto

**Que el script termine no significa que la muestra sea correcta.**

*Ampliación opcional, para quien ya haya terminado o para después de la sesión: pide un segundo script que compruebe automáticamente estas propiedades.*

In [ ]:
ver_excel("MI_MUESTRA_MADRID.xlsx", filas=(1, 33), columnas=4)

---
# REGLAS · ¿Cómo debe calcularse?

*(Diapositivas 32–38)* · 3 / 8 resueltas (OBJETIVO ✓, ESTRUCTURA ✓, RESTRICCIONES ✓)

Ya entendemos los datos y sabemos qué compartir. Ahora: ¿cómo se combinan varias comunidades?

## Aparece una segunda comunidad

Dos entradas limpias, ya revisadas, cuyo contenido conocemos: **Comunidad de Madrid** (`ENTRADAS_LIMPIAS/MADRID.xlsx`) y
**Castilla y León** (`ENTRADAS_LIMPIAS/CASTILLA_Y_LEON.xlsx`).

In [ ]:
ir_a("03_modelo_2_ccaa/ENTRADAS_LIMPIAS")
print("MADRID")
ver_excel("MADRID.xlsx", filas=(5, 9), columnas=4)
print()
print("CASTILLA Y LEÓN")
ver_excel("CASTILLA_Y_LEON.xlsx", filas=(5, 9), columnas=4)

**Si combinamos estas dos comunidades, ¿cómo obtenemos un resultado conjunto?**

## ¿Cómo obtenemos el porcentaje conjunto? · ejemplo simplificado

Usamos números redondos (no los de Madrid y Castilla y León):

- **CCAA A:** Sí 8 de 10 controles → 80 %
- **CCAA B:** Sí 10 de 40 controles → 25 %

Una comunidad tiene 10 controles. La otra tiene 40.

### ¿Qué resultado conjunto elegirías?

**A · 52,5 %**  o  **B · 36 %**

Decide (o vota con tu mesa) antes de seguir.

In [ ]:
#@title Ver la respuesta (ejecútala cuando hayas pensado la tuya)
revelar("""
**52,5 % sale de promediar directamente** ✗

(80 % + 25 %) / 2 = 52,5 %  — ¿por qué es engañoso? **10 controles ≠ 40 controles.**
La comunidad con 40 controles debería pesar más que la de 10.
""")

In [ ]:
#@title Ver la respuesta (ejecútala cuando hayas pensado la tuya)
revelar("""
**Primero sumamos recuentos. Después calculamos porcentajes.** ✓

| | Sí | No | Total |
|---|:-:|:-:|:-:|
| CCAA A | 8 | 2 | 10 |
| CCAA B | 10 | 30 | 40 |

(8 + 10) / (10 + 40) = 18 / 50 = **36 %**

*Un programa puede ejecutar perfectamente una regla equivocada.*
""")

## Tercer mensaje: cómo debe agregarse

Envía el **MENSAJE 3** en la misma conversación:

```
Para obtener el resultado conjunto, suma primero los recuentos Sí, No y No procede de todas las comunidades para cada pregunta y calcula los porcentajes después, sobre esos totales.

No promedies los porcentajes de cada comunidad: las comunidades tienen distinto número de controles.
```

*La regla, escrita con su motivo: así la IA no tiene que elegir.*

> 🟢 **IGUAL** — actividad de pensar (el ejemplo es de papel) y mensaje 3, iguales que en clase.

---
# CONTROLES · ¿Cómo demuestro que funciona?

*(Diapositivas 39–47)* · 4 / 8 resueltas (+ REGLAS ✓)

Ya sabemos calcular. Ahora necesitamos demostrar que un programa hace correctamente ese cálculo.

## Tenemos dos entradas cuyo resultado conocemos

Madrid (entrada limpia, revisada) + Castilla y León (entrada limpia, revisada) + la regla (sumar recuentos → después %) =
**un resultado esperado que conocemos de antemano**: 49 controles en cada pregunta (27 + 22).

**Si ahora hacemos un programa que combine solo estas dos comunidades, ¿cómo demostraríamos que lo está haciendo bien?**

## Mini-práctica · Combina solo dos comunidades

- **Entradas:** `MADRID.xlsx` + `CASTILLA_Y_LEON.xlsx` (`03_modelo_2_ccaa/ENTRADAS_LIMPIAS/`).
- **El script debe:** leer las dos entradas · sumar Sí, No y No procede de ambas en cada pregunta · generar `CONSOLIDADO_2_CCAA` con las 25 preguntas · crear un archivo nuevo · no modificar los originales.
- **Salida mínima:** `CONSOLIDADO_2_CCAA` con *Pregunta · Sí · No · No procede · Total* (25 filas, recuentos conjuntos).

*Solo Madrid y Castilla y León. Sin 17 CCAA, gráficas, ranking ni informes.* Puedes reutilizar los mensajes de estructura y reglas.

Prompt orientativo (de `PROMPTS_REGLAS_Y_CONTROLES.txt`, con las rutas de Colab):

```
Tengo dos archivos limpios, MADRID.xlsx y CASTILLA_Y_LEON.xlsx. Ambos contienen las mismas 25 preguntas y las columnas Pregunta, Sí, No y No procede.

Escribe un script pequeño que genere un Excel nuevo con un consolidado de las 25 preguntas, sumando los recuentos de ambas comunidades. No modifiques los originales. Antes del código, dime qué comprobaciones harías para poder comparar la salida con un resultado esperado independiente.

Lo ejecutaré en Google Colab con "!python mi_combinar_2_ccaa.py" desde la carpeta 03_modelo_2_ccaa/CONTROLES. Los Excel están en ../ENTRADAS_LIMPIAS/. Guarda el resultado en SALIDAS/MI_CONSOLIDADO_2_CCAA.xlsx, en una hoja CONSOLIDADO_2_CCAA con las columnas Pregunta, Sí, No, No procede y Total.
```

> 🟡 **ADAPTADA** — guardas el script con `%%writefile` y lo ejecutas con `!python`.

In [ ]:
ir_a("03_modelo_2_ccaa/CONTROLES")

In [ ]:
%%writefile mi_combinar_2_ccaa.py
# ↓ Borra estas dos líneas y pega aquí, completo, el código que te ha dado tu IA ↓

In [ ]:
!python mi_combinar_2_ccaa.py

## Parece funcionar. ¿Cómo lo sabemos?

El script ha terminado y ha generado un archivo con buen aspecto (primeras filas de **tu** salida):

**¿Qué comprobarías antes de fiarte?** Apunta 2–3 ideas.

In [ ]:
hojas("SALIDAS/MI_CONSOLIDADO_2_CCAA.xlsx")
ver_excel("SALIDAS/MI_CONSOLIDADO_2_CCAA.xlsx", "CONSOLIDADO_2_CCAA", filas=(1, 8), columnas=6)

## Comparamos con un resultado conocido

Compara **tu salida** con el **resultado conocido, independiente** (`RESULTADO_ESPERADO_2_CCAA.xlsx`, calculado a mano /
de una referencia revisada). Mira ahora las 25 filas de las dos y comprueba esta lista:

- □ 25 preguntas
- □ cada pregunta aparece una sola vez
- □ 49 controles en las 25 preguntas (27 + 22)
- □ mismos recuentos que el resultado conocido
- □ Sí + No + No procede = total en cada fila
- □ originales intactos

El cuaderno no lo comprueba por ti: **lo haces tú** (así lo hacías en clase, con las dos tablas delante).

In [ ]:
print("TU SALIDA")
ver_excel("SALIDAS/MI_CONSOLIDADO_2_CCAA.xlsx", "CONSOLIDADO_2_CCAA", filas=(1, 26), columnas=6)

In [ ]:
print("RESULTADO CONOCIDO (independiente)")
ver_excel("RESULTADO_ESPERADO_2_CCAA.xlsx", "CONSOLIDADO_ESPERADO", filas=(1, 26), columnas=6)

In [ ]:
#@title Ver la respuesta (ejecútala cuando hayas pensado la tuya)
revelar("""
Algunos valores de control para contrastar a ojo: **P1 = 24 / 25 / 0** y **P6 = 20 / 6 / 23** (total 49 en las 25 preguntas).

*Cuando puedo construir un caso pequeño cuyo resultado conozco, puedo utilizarlo para probar una automatización.*
""")

## Control 1 · un caso pequeño conocido es una prueba

**Caso pequeño** (entradas que conocemos bien) → **programa** (la automatización que queremos probar) → **salida** → **comparar** con el resultado esperado, calculado de forma independiente.

Si coinciden, ganamos confianza. Si no, sabemos exactamente dónde mirar.
Este control comprueba un resultado **estático**: una entrada conocida, una salida esperada.

## Control 2 · cambio controlado

Las herramientas cambian: se corrige un dato, se modifica el código… **Modificamos deliberadamente UNA cosa** en la entrada de Madrid.

- **ANTES · `MADRID.xlsx` · pregunta 1:** Sí 12 · No 15 · No procede 0
- **DESPUÉS · `MADRID_CAMBIO_CONTROLADO.xlsx` · pregunta 1:** Sí 13 · No 14 · No procede 0
- Total de Madrid: 27 → 27

**Antes de ejecutar nada, PREDICE** (por escrito o en voz alta):

- □ ¿qué fila debe cambiar?
- □ ¿qué agregado conjunto debe cambiar?
- □ ¿qué debe permanecer igual?
- □ ¿qué preguntas no deben verse afectadas?

*Primero predecimos. Después ejecutamos y comparamos.*

In [ ]:
# Los dos Madrid: mira la fila de la pregunta 1 en cada uno
print("ANTES")
ver_excel("../ENTRADAS_LIMPIAS/MADRID.xlsx", filas=(5, 8), columnas=4)
print()
print("DESPUÉS")
ver_excel("CAMBIO_CONTROLADO/MADRID_CAMBIO_CONTROLADO.xlsx", filas=(5, 8), columnas=4)

Para ejecutar tu script con el Madrid modificado **sin editar nada a mano**, tu programa necesita aceptar otro archivo de Madrid y otra salida.
Pídele esto a tu IA (con tu código pegado):

```
Modifica mi script para que acepte dos opciones de línea de comandos opcionales: --madrid (ruta del Excel de Madrid; por defecto ../ENTRADAS_LIMPIAS/MADRID.xlsx) y --salida (ruta del Excel de salida; por defecto SALIDAS/MI_CONSOLIDADO_2_CCAA.xlsx). No cambies nada más.
```

Pega la nueva versión aquí (sustituye a la anterior) y ejecútala **con tu predicción ya hecha**:

> 🟡 **ADAPTADA** — en clase se ejecuta con la entrada ya modificada desde la terminal; aquí el comando va en una celda. (Si tu script ya acepta esas opciones, no hace falta repetir el prompt.)

In [ ]:
%%writefile mi_combinar_2_ccaa.py
# ↓ Borra estas dos líneas y pega aquí, completo, el código que te ha dado tu IA ↓

In [ ]:
!python mi_combinar_2_ccaa.py --madrid CAMBIO_CONTROLADO/MADRID_CAMBIO_CONTROLADO.xlsx --salida CAMBIO_CONTROLADO/MI_SALIDA_CAMBIO.xlsx

### Ejecutamos y comparamos con la predicción

Compara **ANTES** (tu consolidado inicial) con **DESPUÉS** (`MI_SALIDA_CAMBIO.xlsx`) y con
`CAMBIO_CONTROLADO/RESULTADO_ESPERADO_CAMBIO.xlsx`. Debe coincidir en las 25 filas.

In [ ]:
print("ANTES · tu consolidado inicial")
ver_excel("SALIDAS/MI_CONSOLIDADO_2_CCAA.xlsx", "CONSOLIDADO_2_CCAA", filas=(1, 5), columnas=6)
print()
print("DESPUÉS · con el cambio controlado")
ver_excel("CAMBIO_CONTROLADO/MI_SALIDA_CAMBIO.xlsx", "CONSOLIDADO_2_CCAA", filas=(1, 5), columnas=6)
print()
print("RESULTADO ESPERADO DEL CAMBIO")
ver_excel("CAMBIO_CONTROLADO/RESULTADO_ESPERADO_CAMBIO.xlsx", filas=(1, 5), columnas=6)

In [ ]:
#@title Ver la respuesta (ejecútala cuando hayas pensado la tuya)
revelar("""
**DEBE CAMBIAR**

- la fila de Madrid · pregunta 1
- Sí conjunto de la pregunta 1: **24 → 25**
- No conjunto de la pregunta 1: **25 → 24**
- los porcentajes de la pregunta 1

**DEBE PERMANECER IGUAL**

- ✓ total de la pregunta 1: 49 → 49
- ✓ No procede de la pregunta 1: 0 → 0
- ✓ preguntas 2–25
- ✓ todos los datos de Castilla y León

*Un buen control no solo comprueba un resultado estático: también permite comprobar que un cambio produce exactamente los efectos esperados. A esto se le suele llamar prueba de regresión.*
""")

## Dos formas de controlar una automatización

> 🟢 **IGUAL** — síntesis conceptual, igual que en clase.

| CONTROL 1 · CASO CONOCIDO | CONTROL 2 · CAMBIO CONTROLADO |
|---|---|
| 1. entrada conocida | 1. modifico UNA cosa |
| 2. programa | 2. predigo qué cambia y qué no |
| 3. salida | 3. ejecuto |
| 4. comparar con referencia independiente | 4. comparo con la predicción |

---
# RESULTADO · ¿Qué debe entregar?

*(Diapositivas 48–51)* · 5 / 8 resueltas

Ya hemos probado el núcleo. Ahora concretamos qué artefactos debe producir la herramienta.

## El modelo funcional completo

Abre `03_modelo_2_ccaa/01_MODELO_REFORZADO_2_CCAA.xlsx` («modelo funcional preparado para la tarea»). Tiene 7 hojas:

**Datos por comunidad** (`MADRID_DATOS` / `CASTILLA_Y_LEON_DATOS`) → **Estadísticas por comunidad** (`MADRID_ESTADISTICAS` / `CASTILLA_Y_LEON_EST`) → **Consolidado** (`CONSOLIDADO_2_CCAA`) → **Estadísticas conjuntas** (`ESTADISTICAS_2_CCAA` + 4 gráficas) → **Control** (`CONTROL`).

> 🟡 **ADAPTADA** — abrir el modelo con Excel → `ver_excel`. Las **gráficas** no se pueden ver con pandas: aquí se listan, y `descargar(...)` te permite abrirlas en tu equipo (Excel, LibreOffice o Google Sheets).

In [ ]:
ir_a("03_modelo_2_ccaa")
MODELO = "01_MODELO_REFORZADO_2_CCAA.xlsx"
hojas(MODELO)

In [ ]:
ver_excel(MODELO, "CONSOLIDADO_2_CCAA", filas=(5, 9), columnas=4)

In [ ]:
ver_excel(MODELO, "ESTADISTICAS_2_CCAA", filas=(1, 8), columnas=8)

In [ ]:
ver_excel(MODELO, "CONTROL", filas=(1, 4), columnas=7)

In [ ]:
# Las 4 gráficas de ESTADISTICAS_2_CCAA (títulos). Para verlas dibujadas, descarga el archivo y ábrelo en tu equipo.
graficos(MODELO)
# descargar(MODELO)   # ← quita el # y ejecuta para descargarlo

**¿Qué queremos que entregue nuestra herramienta?** Apunta tus respuestas y contrástalas con los entregables:

## Estos son nuestros entregables

| ENTREGABLE | QUÉ CONTIENE | REFERENCIA EQUIVALENTE EN EL MODELO |
|---|---|---|
| `DATOS_CONSOLIDADOS` | una fila por comunidad y pregunta, con archivo y hoja de origen | `MADRID_DATOS` · `CASTILLA_Y_LEON_DATOS` |
| `ESTADISTICAS_CCAA` | recuentos y porcentajes de cada comunidad | `MADRID_ESTADISTICAS` · `CASTILLA_Y_LEON_EST` |
| `NACIONAL` | resultado conjunto calculado sobre la suma de recuentos | `CONSOLIDADO_2_CCAA` · `ESTADISTICAS_2_CCAA` |
| resumen por bloques | Cumple / Incumple / No procede de cada bloque de preguntas | filas de bloque de `ESTADISTICAS_2_CCAA` |
| 4 gráficas | una general por bloques y una por cada bloque | las 4 gráficas de `ESTADISTICAS_2_CCAA` |
| `CONTROL_CARGA` | qué se ha leído de cada archivo y si hay incidencias | hoja `CONTROL` |

*El modelo no es una entrada: es el ejemplo que nos ayuda a decidir qué pedir.*

> 🟢 **IGUAL** — la tabla de entregables y el mensaje 4 son los mismos que en clase.

## Cuarto mensaje: qué debe generar

Envía el **MENSAJE 4** y **adjunta** `01_MODELO_REFORZADO_2_CCAA.xlsx` como ejemplo (es una referencia, no una entrada de datos):

```
Quiero que el programa genere un Excel nuevo con:

- DATOS_CONSOLIDADOS: una fila por comunidad y pregunta, indicando archivo y hoja de origen;
- ESTADISTICAS_CCAA: recuentos y porcentajes por comunidad;
- NACIONAL: resultado estatal calculado sobre la suma de recuentos;
- un resumen por bloques de preguntas;
- cuatro gráficas nacionales: una general por bloques y una por cada bloque;
- CONTROL_CARGA: qué se ha leído de cada archivo y si hay incidencias.

Adjunto 01_MODELO_REFORZADO_2_CCAA.xlsx como ejemplo pequeño de la estructura de las principales salidas. Es una referencia, no una entrada de datos.
```

*Un ejemplo adjunto vale más que un párrafo describiendo formatos.*

> 🟡 **ADAPTADA** — para adjuntarlo, descárgalo con la celda siguiente y súbelo al chat de tu IA (con datos sintéticos del curso no hay problema; con datos reales dependería del entorno autorizado).

In [ ]:
descargar(BASE / "03_modelo_2_ccaa" / "01_MODELO_REFORZADO_2_CCAA.xlsx")

---
# ARCHIVOS · ¿Qué fuentes concretas procesa?

*(Diapositivas 52–56)* · 6 / 8 resueltas

**Parece que ya lo tenemos…** Seis piezas resueltas. Pero hasta ahora solo hemos trabajado con **dos entradas limpias**.

## Esto es lo que tenemos que integrar

- **ENTRADAS:** 1 libro con 12 CCAA ya integradas + 5 archivos pendientes (uno por comunidad) = **17 CCAA** a consolidar.
- **REFERENCIA:** el modelo funcional de 2 CCAA (Madrid + Castilla y León). Sirve para comparar, no para sumar.

**El modelo de 2 CCAA no es una entrada.** Añadirlo duplicaría Madrid y Castilla y León, que ya están dentro del libro de 12 CCAA.

## Antes de escribir código: revisa los materiales

**TODAVÍA NO PROGRAMES.** Abre:

- `05_reto_gimnasios/02_BASE_12_CCAA_INTEGRADAS.xlsx`
- `05_reto_gimnasios/CCAA_PENDIENTES/` · los 5 archivos
- `03_modelo_2_ccaa/01_MODELO_REFORZADO_2_CCAA.xlsx`, si quieres compararlo

Pregúntate: ¿qué parece estable? ¿qué cambia? ¿qué hojas contienen datos? ¿qué supuestos serían peligrosos?
**Diagnóstico antes de solución.**

> 🟡 **ADAPTADA** — abrir los Excel con Excel → `hojas` y `ver_excel`. Los archivos de `CCAA_PENDIENTES` los verás uno a uno en la sección siguiente.

In [ ]:
ir_a("05_reto_gimnasios")
hojas("02_BASE_12_CCAA_INTEGRADAS.xlsx")

In [ ]:
ver_excel("02_BASE_12_CCAA_INTEGRADAS.xlsx", "INDICE", filas=(1, 13), columnas=7)

In [ ]:
# Los cinco archivos pendientes: qué hojas tiene cada uno
for f in sorted(Path("CCAA_PENDIENTES").glob("*.xlsx")):
    hojas(f)

### Primera conclusión de la revisión

Los cinco archivos pendientes (`ARAGON`, `ASTURIAS`, `GALICIA`, `MURCIA`, `COMUNITAT_VALENCIANA`) **pretenden representar lo mismo**:
25 preguntas · `Pregunta | Sí | No | No procede`… **pero no todos lo presentan exactamente de la misma forma.**

Tenemos un significado común. Ahora vamos a ver qué cambia en su representación.

---
# VARIACIONES · ¿Qué diferencias debe tolerar?

*(Diapositivas 57–73)* · 7 / 8 resueltas

Los cinco pendientes, uno a uno, siempre junto al mismo **archivo normal: Madrid** (hoja `Madrid` de `02_BASE_12_CCAA_INTEGRADAS.xlsx`).

> 🟡 **ADAPTADA** — en clase se proyectan los dos Excel; aquí los ves uno debajo del otro con `ver_excel` (con números de fila y letras de columna, para localizar `B6`, la fila 10…)
Para cada caso: **observa** y piensa *«¿qué cambia respecto al archivo normal?»*; después ejecuta la celda de **revelación**.

*Cada archivo pendiente trae también una hoja `LEEME`; los datos están en la hoja con el nombre de la comunidad.*

## Caso 1 de 5 · Aragón · observa

¿Qué cambia respecto al archivo normal (Madrid)?

In [ ]:
print("ARCHIVO NORMAL · Madrid · 02_BASE_12_CCAA_INTEGRADAS.xlsx")
ver_excel("02_BASE_12_CCAA_INTEGRADAS.xlsx", "Madrid", filas=(1, 12), columnas=5)
print()
print("CCAA_PENDIENTES/ARAGON.xlsx")
ver_excel("CCAA_PENDIENTES/ARAGON.xlsx", "Aragón", filas=(1, 12), columnas=5)

### Caso 1 de 5 · Aragón · revelación

Ejecuta la celda cuando hayas formulado tu respuesta.

In [ ]:
#@title Ver la respuesta (ejecútala cuando hayas pensado la tuya)
revelar("""
**QUÉ OCURRE:** la tabla está **desplazada**. La cabecera `Pregunta` está en **B6** (en Madrid, en A5): hay una columna A vacía y la cabecera baja a la fila 6.

**SUPUESTO QUE ROMPE:** no podemos asumir una posición fija para la cabecera.

*Regla que nos llevamos: la cabecera hay que BUSCARLA.*
""")

## Caso 2 de 5 · Asturias · observa

¿Qué cambia respecto al archivo normal (Madrid)?

In [ ]:
print("ARCHIVO NORMAL · Madrid · 02_BASE_12_CCAA_INTEGRADAS.xlsx")
ver_excel("02_BASE_12_CCAA_INTEGRADAS.xlsx", "Madrid", filas=(1, 12), columnas=4)
print()
print("CCAA_PENDIENTES/ASTURIAS.xlsx")
ver_excel("CCAA_PENDIENTES/ASTURIAS.xlsx", "Asturias", filas=(1, 14), columnas=4)

### Caso 2 de 5 · Asturias · revelación

Ejecuta la celda cuando hayas formulado tu respuesta.

In [ ]:
#@title Ver la respuesta (ejecútala cuando hayas pensado la tuya)
revelar("""
**QUÉ OCURRE:** **filas vacías intercaladas** entre las preguntas (por ejemplo, la fila 10); la hoja llega hasta la fila 44 en lugar de 40.

**SUPUESTO QUE ROMPE:** una fila vacía no puede significar automáticamente fin de tabla. Leer «hasta el primer hueco» perdería las preguntas posteriores.

*Regla: localizar las preguntas 1–25, no detenerse en el primer vacío.*
""")

## Caso 3 de 5 · Galicia · observa

¿Qué cambia respecto al archivo normal (Madrid)?

In [ ]:
print("ARCHIVO NORMAL · Madrid · 02_BASE_12_CCAA_INTEGRADAS.xlsx")
ver_excel("02_BASE_12_CCAA_INTEGRADAS.xlsx", "Madrid", filas=(1, 12), columnas=4)
print()
print("CCAA_PENDIENTES/GALICIA.xlsx")
ver_excel("CCAA_PENDIENTES/GALICIA.xlsx", "Galicia", filas=(1, 12), columnas=4)

### Caso 3 de 5 · Galicia · revelación

Ejecuta la celda cuando hayas formulado tu respuesta.

In [ ]:
#@title Ver la respuesta (ejecútala cuando hayas pensado la tuya)
revelar("""
**QUÉ OCURRE:** **columnas reordenadas**. El orden es `Pregunta · No procede · Sí · No` (en Madrid: `Sí · No · No procede`).

**SUPUESTO QUE ROMPE:** no podemos identificar Sí / No / No procede por número de columna: leer por posición sumaría la columna equivocada… sin ningún error visible.

*Regla: identificar cada columna por su NOMBRE.*
""")

## Caso 4 de 5 · Murcia · observa

¿Qué cambia respecto al archivo normal (Madrid)?

In [ ]:
print("ARCHIVO NORMAL · Madrid · 02_BASE_12_CCAA_INTEGRADAS.xlsx")
ver_excel("02_BASE_12_CCAA_INTEGRADAS.xlsx", "Madrid", filas=(1, 12), columnas=5)
print()
print("CCAA_PENDIENTES/MURCIA.xlsx")
ver_excel("CCAA_PENDIENTES/MURCIA.xlsx", "Murcia", filas=(1, 14), columnas=5)

### Caso 4 de 5 · Murcia · revelación

Ejecuta la celda cuando hayas formulado tu respuesta.

In [ ]:
#@title Ver la respuesta (ejecútala cuando hayas pensado la tuya)
revelar("""
**QUÉ OCURRE:** **desplazamiento + filas vacías + columnas reordenadas**. Combina las tres variaciones anteriores: cabecera en B6 (tabla desplazada), filas vacías intercaladas (por ejemplo, la fila 11) y columnas `No procede · Sí · No`.

**SUPUESTO QUE ROMPE:** resolver solo una excepción no basta; hace falta una estrategia que funcione con las tres a la vez.
""")

## Caso 5 de 5 · Comunitat Valenciana · observa

¿Qué cambia respecto al archivo normal (Madrid)?

In [ ]:
print("ARCHIVO NORMAL · Madrid · 02_BASE_12_CCAA_INTEGRADAS.xlsx")
ver_excel("02_BASE_12_CCAA_INTEGRADAS.xlsx", "Madrid", filas=(1, 12), columnas=4)
print()
print("CCAA_PENDIENTES/COMUNITAT_VALENCIANA.xlsx")
ver_excel("CCAA_PENDIENTES/COMUNITAT_VALENCIANA.xlsx", "C. Valenciana", filas=(1, 12), columnas=4)

### Caso 5 de 5 · Comunitat Valenciana · revelación

Ejecuta la celda cuando hayas formulado tu respuesta.

In [ ]:
#@title Ver la respuesta (ejecútala cuando hayas pensado la tuya)
revelar("""
**QUÉ OCURRE:** aparentemente, **nada relevante en la estructura**.

**LO QUE COMPROBAMOS:** cabecera, preguntas y columnas están donde esperamos (cabecera `Pregunta · Sí · No · No procede` en A5; preguntas desde la fila 7; sin filas vacías intercaladas).

*Es una conclusión correcta: a nivel ESTRUCTURAL, Comunitat Valenciana es normal. Ahora estamos mirando la estructura.*
""")

## Busca significado, no coordenadas

1. **Encontrar la cabecera** *(Aragón)*
2. **Identificar columnas por su nombre** *(Galicia)*
3. **Localizar preguntas 1–25** sin parar en huecos *(Asturias)*
4. **Ignorar lo que no pertenece a la estructura** *(filas ajenas y hojas auxiliares)*

*Murcia = las tres primeras a la vez. Comunitat Valenciana: ya encaja → se traslada tal cual.*

**EVITA:** fila 6 · columna B  ·  **BUSCA:** Pregunta · Sí · No · No procede.
**La posición cambia. El significado permanece.**

## Prompt · construye el normalizador

Casos concretos → regla general → instrucciones para la IA → script. Este es el prompt (primero de `NORMALIZACION/PROMPTS_NORMALIZACION.txt`), con una última línea para Colab:

```
Necesito un script que normalice la estructura de los 5 archivos pendientes: ARAGON, ASTURIAS, GALICIA, MURCIA y COMUNITAT_VALENCIANA.

Para cada archivo:
- localiza la tabla buscando sus encabezados, no con coordenadas fijas;
- identifica las columnas Pregunta, Sí, No y No procede por su nombre;
- localiza las preguntas 1–25;
- ignora filas vacías y elementos ajenos a la tabla;
- genera una copia nueva con la estructura común Pregunta | Sí | No | No procede;
- conserva exactamente los valores originales: no conviertas tipos ni corrijas valores;
- si el archivo ya tiene la estructura esperada, traslada sus valores sin alterarlos;
- no modifiques los originales.

El objetivo es normalizar la ESTRUCTURA, no validar ni corregir el contenido.

Lo ejecutaré en Google Colab con "!python mi_normalizar_entradas.py" desde la carpeta 05_reto_gimnasios/NORMALIZACION. Los archivos pendientes están en ../CCAA_PENDIENTES/. Guarda las copias en SALIDAS_NORMALIZADAS/ con el nombre <NOMBRE>_NORMALIZADO.xlsx (por ejemplo ARAGON_NORMALIZADO.xlsx), con los datos en una hoja llamada DATOS.
```

## Mini-práctica · Normaliza las entradas (de cinco formatos a una estructura común)

- **El script debe:** localizar la tabla y su cabecera · identificar las columnas por su nombre · localizar las preguntas 1–25 · ignorar filas vacías y elementos ajenos · conservar los valores tal como llegan (sin convertir tipos) · copias nuevas, originales intactos.
- **Salida:** `SALIDAS_NORMALIZADAS/` con 5 archivos `*_NORMALIZADO.xlsx`; cada uno con 25 filas `Pregunta | Sí | No | No procede`.

*Solo estructura. Sin convertir valores. Sin estadísticas, consolidado ni gráficas.*

> 🟡 **ADAPTADA** — guardas el script con `%%writefile` y lo ejecutas con `!python`.

In [ ]:
ir_a("05_reto_gimnasios/NORMALIZACION")

In [ ]:
%%writefile mi_normalizar_entradas.py
# ↓ Borra estas dos líneas y pega aquí, completo, el código que te ha dado tu IA ↓

In [ ]:
!python mi_normalizar_entradas.py

In [ ]:
# Abre una de tus salidas: Aragón, pregunta 1 (en el original de Aragón era 8 / 10 / 0)
!ls SALIDAS_NORMALIZADAS
ver_excel("SALIDAS_NORMALIZADAS/ARAGON_NORMALIZADO.xlsx", filas=(1, 6), columnas=4)

## ¿Hemos perdido o cambiado algo al normalizar?

**¿Cómo sabemos que no hemos perdido o cambiado datos al normalizar?**
Aplica lo aprendido en CONTROLES: ¿con qué resultado conocido podemos comparar? (Con el archivo original de cada comunidad.)

*Contraste útil: en la muestra sintética queríamos que los recuentos CAMBIARAN; aquí queremos que NO cambien.*

## Prompt · construye un comprobador independiente

Un segundo script, **independiente del normalizador**, que compare cada original con su versión normalizada
(segundo prompt de `PROMPTS_NORMALIZACION.txt`, con una última línea para Colab):

```
Escribe un segundo script, independiente del normalizador, que compare cada archivo original con su versión normalizada.

Debe comprobar:
- que existen los cinco archivos de salida;
- que cada uno contiene 25 preguntas;
- que las preguntas 1–25 aparecen una sola vez;
- que los textos de las preguntas se conservan;
- que los valores Sí / No / No procede son exactamente los del origen;
- que se conservan los totales por pregunta;
- que los originales no se han modificado.

Genera CONTROL_NORMALIZACION.xlsx con una fila por archivo y un estado OK / REVISAR.

No conviertas tipos ni evalúes todavía si los valores son numéricos. Esta comprobación solo responde a una pregunta: ¿la normalización estructural ha conservado correctamente los datos?

Lo ejecutaré en Google Colab con "!python mi_comprobar_normalizacion.py" desde la carpeta 05_reto_gimnasios/NORMALIZACION. Los originales están en ../CCAA_PENDIENTES/, los normalizados en SALIDAS_NORMALIZADAS/, y el informe debe guardarse en SALIDAS_NORMALIZADAS/CONTROL_NORMALIZACION.xlsx.
```

*Un segundo script comprueba el trabajo del primero.*

In [ ]:
%%writefile mi_comprobar_normalizacion.py
# ↓ Borra estas dos líneas y pega aquí, completo, el código que te ha dado tu IA ↓

In [ ]:
!python mi_comprobar_normalizacion.py

In [ ]:
ver_excel("SALIDAS_NORMALIZADAS/CONTROL_NORMALIZACION.xlsx", filas=(1, 8), columnas=12)

In [ ]:
#@title Ver la respuesta (ejecútala cuando hayas pensado la tuya)
revelar("""
**Normalizar no es alterar.** Cuando tu comprobador funciona, deberías ver:

- ✓ 5 archivos de salida · ✓ 25 preguntas por archivo · ✓ 1–25 una sola vez
- ✓ textos de las preguntas · ✓ mismos valores que en el origen · ✓ mismos tipos
- ✓ mismos totales por pregunta · ✓ originales intactos
- estado **OK** en los cinco, **incluida Comunitat Valenciana**

**OK significa:** la normalización no ha perdido ni alterado datos.
*Un control responde a la pregunta para la que fue diseñado.* Este no responde todavía a otras preguntas sobre los datos
(por ejemplo, si los valores son numéricos): lo recuperaremos en la auditoría.

Puedes reutilizar el normalizador como contexto en el reto completo. Es opcional.
""")

---
# Ahora ya tenemos la información que al principio faltaba

*(Diapositivas 74–80)* · **8 / 8 resueltas**

**OBJETIVO** (visión estatal regenerable) · **ESTRUCTURA** (25 preguntas, Sí/No/No procede) · **RESTRICCIONES** (mínima información, muestra sintética) · **REGLAS** (sumar recuentos → después porcentajes) · **CONTROLES** (casos pequeños conocidos, cambio controlado) · **RESULTADO** (entregables con un modelo funcional) · **ARCHIVOS** (12 + 5; el modelo queda fuera) · **VARIACIONES** (desplazamientos, huecos, columnas, combinación).

Recuerda el encargo inicial: *«Hazme un programa que junte estos Excel, saque estadísticas y gráficas.»*
Y las respuestas que apuntaste: ¿cuáles acertaste? ¿cuáles no se te ocurrieron?

## Ya no partimos de cero

Hemos decidido las ocho piezas, y además tenemos: 📎 el modelo funcional de 2 CCAA · 🐍 el normalizador (opcional) · casos pequeños cuyo resultado conocemos · una forma de probar cambios controlados.
*Ahora no necesitamos inventar un mega-prompt desde cero. Podemos ensamblar lo que ya sabemos.*

## El encargo también son los adjuntos

- **ENTRADAS:** 📎 `02_BASE_12_CCAA_INTEGRADAS.xlsx` · 📎 `CCAA_PENDIENTES/` (5 archivos)
- **REFERENCIA · NO ES ENTRADA:** 📎 `01_MODELO_REFORZADO_2_CCAA.xlsx`
- **CÓDIGO REUTILIZABLE · OPCIONAL:** 📎 `normalizar_entradas.py` (o el tuyo)

*El contexto para la IA no son solo palabras: también archivos, ejemplos y código que ya hemos probado.*

> 🟡 **ADAPTADA** — en clase los adjuntas desde tu carpeta; aquí los descargas con la celda siguiente y los subes al chat de tu IA (solo datos sintéticos del curso).

In [ ]:
ir_a("05_reto_gimnasios")
descargar(BASE / "05_reto_gimnasios" / "02_BASE_12_CCAA_INTEGRADAS.xlsx")
descargar_carpeta(BASE / "05_reto_gimnasios" / "CCAA_PENDIENTES")     # un ZIP con los 5 archivos; descomprímelo antes de subirlos
descargar(BASE / "03_modelo_2_ccaa" / "01_MODELO_REFORZADO_2_CCAA.xlsx")
# Opcional: tu normalizador → descargar(BASE / "05_reto_gimnasios" / "NORMALIZACION" / "mi_normalizar_entradas.py")

## Mensaje 5 · primer hito operativo

> 🟢 **IGUAL** — el mensaje es el mismo que en clase.

Con los adjuntos anteriores, envía el **MENSAJE 5** (idéntico al de `prompt_construccion_guiada.txt`):

```
Con los archivos de entrada adjuntos, construye una primera versión de la herramienta.

En esta iteración quiero únicamente:

- leer correctamente el libro con 12 CCAA y los 5 archivos pendientes;
- localizar las tablas por sus encabezados, no por coordenadas fijas;
- extraer las preguntas 1–25;
- conservar comunidad, archivo y hoja de origen;
- generar DATOS_CONSOLIDADOS;
- generar CONTROL_CARGA con evidencia de qué se ha leído y qué se ha podido interpretar.

Todavía no generes estadísticas ni gráficas.

No modifiques los originales.

Si te resulta útil, puedes reutilizar la lógica de normalizar_entradas.py como contexto.

Antes del código, resume los controles que incluirás y los supuestos que estás haciendo.
```

*Una primera versión limitada: leer, reunir y dejar constancia de lo que se ha leído.*

## Normalizar ≠ consolidar

| NORMALIZAR | CONSOLIDAR |
|---|---|
| trabaja con **cada archivo individual** (Aragón desplazado → Aragón con estructura común) | trabaja con **varias fuentes a la vez** → una tabla común con el origen de cada registro |
| ¿Puedo interpretar distintas fuentes con la misma estructura? | ¿Puedo reunir todos los registros conservando su origen? |

*Un mismo script puede normalizar internamente y después consolidar.*

## Primer hito · una base común

Demo de `04_construccion_guiada/01_CONSOLIDAR_BASE.py`: lee las 12 CCAA + los 5 pendientes y genera `DATOS_CONSOLIDADOS` + `CONTROL_CARGA`.
*Todavía no queremos estadísticas ni gráficas. Primero quiero demostrar que puedo leer las fuentes y reunirlas.*

> 🟡 **ADAPTADA** — en clase lo ejecuta el docente desde la terminal; aquí lo ejecutas tú con `!python`.

In [ ]:
ir_a("04_construccion_guiada")
!python 01_CONSOLIDAR_BASE.py

In [ ]:
# DATOS_CONSOLIDADOS: una fila por comunidad y pregunta, con archivo y hoja de origen (filas de Andalucía)
ver_excel(BASE / "05_reto_gimnasios/SALIDAS/CONSOLIDADO_BASE.xlsx", "DATOS_CONSOLIDADOS", filas=(1, 4), columnas=11)

In [ ]:
# CONTROL_CARGA: qué ha leído la herramienta de cada fuente (filas 1–10 por ahora)
ver_excel(BASE / "05_reto_gimnasios/SALIDAS/CONSOLIDADO_BASE.xlsx", "CONTROL_CARGA", filas=(1, 10), columnas=9)

## Ya tenemos · todavía falta

| YA TENEMOS | TODAVÍA FALTA |
|---|---|
| ✓ lectura de estructuras diferentes | ○ estadísticas por CCAA |
| ✓ consolidación | ○ resultado nacional |
| ✓ trazabilidad de carga | ○ resumen por bloques |
| | ○ gráficas |
| | ○ controles más profundos |

**Ahora toca completar la herramienta.**

---
# Reto · campaña estatal

*(Diapositiva 81)* · **20 min** · trabajo autónomo

**1 · GENERA** un libro con: `DATOS_CONSOLIDADOS` · `ESTADISTICAS_CCAA` · `NACIONAL` · resumen por bloques · cuatro gráficas · `CONTROL_CARGA`.

**2 · COMPRUEBA:** 17 CCAA · 25 preguntas por CCAA · 425 combinaciones · valores numéricos válidos · Sí + No + No procede = controles · nacional calculado sobre recuentos.

**3 · ESCALA** *(opcional, solo después de comprobar)*: gráficas por CCAA · ranking · índice · incidencias.

**Puedes partir de** `01_CONSOLIDAR_BASE.py` · tu código anterior (normalizador, combinar_2_ccaa) · nada: no hay una única arquitectura correcta.

**Regla de seguridad:** no modifiques, renombres ni sobrescribas los archivos originales. Genera salidas nuevas.
Enunciado completo: `05_reto_gimnasios/ENUNCIADO_RETO_BLOQUE_2.md`.

Trabaja en tu conversación con la IA (con los mensajes 1–5 ya enviados). Para que el resto del cuaderno encuentre tu resultado,
añade esta línea al final de tu último mensaje:

```
Lo ejecutaré en Google Colab con "!python mi_herramienta.py" desde la carpeta 05_reto_gimnasios (las rutas son relativas a esa carpeta). Guarda el resultado en SALIDAS/MI_RESULTADO_17_CCAA.xlsx.
```

Pega el código en la celda siguiente y ejecútala.

> 🟡 **ADAPTADA** — guardas el script con `%%writefile` y lo ejecutas con `!python`; las gráficas del Excel se ven descargando el archivo.

In [ ]:
ir_a("05_reto_gimnasios")

In [ ]:
%%writefile mi_herramienta.py
# ↓ Borra estas dos líneas y pega aquí, completo, el código que te ha dado tu IA ↓

In [ ]:
!python mi_herramienta.py

**Checkpoint · minuto 10:** ¿hay primera salida? ¿aparecen 17 CCAA? ¿has abierto `CONTROL_CARGA`? ¿estás corrigiendo formato antes que datos?

Mira las hojas de **tu** libro:

In [ ]:
MI_SALIDA = BASE / "05_reto_gimnasios" / "SALIDAS" / "MI_RESULTADO_17_CCAA.xlsx"
hojas(MI_SALIDA)

In [ ]:
# Si tu libro tiene gráficas: aquí aparecen sus títulos. Para verlas dibujadas, descárgalo.
graficos(MI_SALIDA)
# descargar(MI_SALIDA)   # ← quita el # y ejecuta para descargarlo

---
# AUDITORÍA · revisar la salida

*(Diapositivas 82–111)* · **El programa ha terminado. El trabajo, no.**

Si hubiéramos elaborado este informe a mano, lo revisaríamos antes de utilizarlo. Automatizar no elimina esa revisión: cambia cómo la hacemos.

- **ANTES · CONTROLES:** probar un script pequeño mientras lo construíamos.
- **AHORA · APLICAMOS Y PROFUNDIZAMOS CONTROLES:** revisar la salida completa de la herramienta.

## ¿Qué puede haber ocurrido?

1. **FUNCIONA CORRECTAMENTE** — termina y supera los controles relevantes.
2. **ERROR VISIBLE** — el programa se detiene.
3. **RESULTADO INCOMPLETO** — termina, pero falta algo detectable.
4. **PLAUSIBLE PERO INCORRECTO** — termina, parece razonable, pero ha interpretado o calculado algo mal.

*El cuarto caso es el más peligroso: no se reconoce a simple vista.* **¿En cuál está tu herramienta?**
> 🟢 **IGUAL** — conceptual, como en clase.

## Primer control · cobertura

✓ 17 CCAA · ✓ 425 registros. Esto descarta algunos fallos. **No demuestra que estemos en el caso 1.**
¿Basta con esto para firmar el resultado?

In [ ]:
# Cobertura de tu libro (hoja DATOS_CONSOLIDADOS): ¿cuántas comunidades y cuántos registros?
import pandas as pd
try:
    d = pd.read_excel(MI_SALIDA, sheet_name="DATOS_CONSOLIDADOS")
    print("registros:", len(d), "· comunidades distintas:", d.iloc[:, 0].nunique())
except Exception as e:
    print("No he podido leer DATOS_CONSOLIDADOS de tu libro:", e)

## La herramienta también debe decirnos qué ha utilizado

`CONTROL_CARGA`: cada fila nos dice qué ha leído la herramienta de cada fuente.
**¿Qué esperaríamos ver en todas?** (columnas `preguntas_detectadas`, `preguntas_numericas_validas`, `controles_realizados` y `estado`)

Primero, la primera base de la demo (`CONSOLIDADO_BASE.xlsx`, filas normales), y después **tu** libro:

In [ ]:
ver_excel(BASE / "05_reto_gimnasios/SALIDAS/CONSOLIDADO_BASE.xlsx", "CONTROL_CARGA", filas=(1, 10), columnas=9)

In [ ]:
print("TU LIBRO · CONTROL_CARGA")
ver_excel(MI_SALIDA, "CONTROL_CARGA", filas=(1, 20), columnas=9)

### Detectar no es utilizar

Ahora mira la fila de **Comunitat Valenciana** en el `CONTROL_CARGA` de la primera base. En clase se descubre en la fila 11.
Ejecuta la celda solo cuando hayas revisado las filas normales:

In [ ]:
print("PRIMERA BASE · fila de Comunitat Valenciana")
ver_excel(BASE / "05_reto_gimnasios/SALIDAS/CONSOLIDADO_BASE.xlsx", "CONTROL_CARGA", buscar="Valenciana", columnas=9)
print()
print("TU LIBRO · fila de Comunitat Valenciana")
ver_excel(MI_SALIDA, "CONTROL_CARGA", buscar="Valenciana", columnas=9)

In [ ]:
#@title Ver la respuesta (ejecútala cuando hayas pensado la tuya)
revelar("""
**Comunitat Valenciana · 25 detectadas · 0 válidas** (dato del `CONTROL_CARGA` que genera la propia herramienta, en la primera base).

El programa termina, genera el Excel y cifras plausibles, pero **esos datos no participan en el cálculo**. **Detectar no es utilizar.**

Si en tu libro salió *25 / 25*: tus soluciones son más robustas que la primera base limitada. Si salió *25 / 0*: acabas de descubrirlo.
¿Quién lo había visto en su `CONTROL_CARGA`?
""")

## De la evidencia a la corrección

No: *«algo falla» → «arréglalo»*. Pedir un arreglo sin diagnóstico suele producir cambios grandes, innecesarios o que esconden el problema.

Sí: **EVIDENCIA → DIAGNÓSTICO → INFORMACIÓN PARA LA IA → CAMBIO CONCRETO → COMPROBACIÓN.**
Es el mismo método de todo el curso, ahora aplicado a un fallo real. Ya tenemos la evidencia.

## PROMPT 1 · Antes de corregir: diagnostica

Envía a tu IA (es el PROMPT 1 de `06_diagnostico_y_validacion/PROMPTS_DIAGNOSTICO_Y_CORRECCION.txt`):

```
El programa termina sin error y genera 17 CCAA y 425 registros. Sin embargo, en CONTROL_CARGA, Comunitat Valenciana muestra 25 preguntas detectadas pero 0 valores numéricamente válidos. Al abrir el Excel, los valores parecen números.

No modifiques todavía el programa. Indícame:
- qué hipótesis comprobarías;
- qué comprobación mínima harías;
- y qué evidencia necesitarías para identificar la causa.
```

*Primero aportamos evidencia. Después pedimos diagnóstico. Corregimos solo cuando conocemos la causa.*

> 🟢 **IGUAL** — el prompt 1 va a tu IA, igual que en clase.

## Mini-práctica · ¿Qué está leyendo realmente Python?

**NO ARREGLES TODAVÍA.** Pide a tu IA un script pequeño que revise las columnas Sí / No / No procede de cada comunidad y muestre:
✓ cuántos valores son numéricos · ✓ cuántos son texto · ✓ cuántos están vacíos · ✓ uno o dos ejemplos: valor + tipo.

Prompt orientativo (deriva de la diapositiva; ajústalo a tu estilo):

```
Escribe un script pequeño que NO corrija nada. Debe revisar, para cada comunidad, las columnas Sí / No / No procede y mostrar:
- cuántos valores son numéricos;
- cuántos son texto;
- cuántos están vacíos;
- uno o dos ejemplos: valor + tipo.
Las entradas son ../05_reto_gimnasios/02_BASE_12_CCAA_INTEGRADAS.xlsx (una hoja por comunidad) y los cinco archivos de ../05_reto_gimnasios/CCAA_PENDIENTES/. No modifiques ningún archivo.

Lo ejecutaré en Google Colab con "!python mi_diagnosticar_tipos.py" desde la carpeta 06_diagnostico_y_validacion.
```

> 🟡 **ADAPTADA** — guardas el script con `%%writefile` y lo ejecutas con `!python`.

In [ ]:
ir_a("06_diagnostico_y_validacion")

In [ ]:
%%writefile mi_diagnosticar_tipos.py
# ↓ Borra estas dos líneas y pega aquí, completo, el código que te ha dado tu IA ↓

In [ ]:
!python mi_diagnosticar_tipos.py

*Separar diagnóstico de corrección.* Ahora la causa es una evidencia, no una suposición.

## En Excel parecen iguales

Mira **la pregunta 1, celda B7 (Sí)** de Madrid y de Comunitat Valenciana, tal como se ven en Excel:

In [ ]:
print("MADRID · pregunta 1 (fila 7)")
ver_excel(BASE / "05_reto_gimnasios/02_BASE_12_CCAA_INTEGRADAS.xlsx", "Madrid", filas=(5, 7), columnas=4)
print()
print("COMUNITAT VALENCIANA · pregunta 1 (fila 7)")
ver_excel(BASE / "05_reto_gimnasios/CCAA_PENDIENTES/COMUNITAT_VALENCIANA.xlsx", "C. Valenciana", filas=(5, 7), columnas=4)

**¿Qué lee realmente el programa?** Ejecuta la celda:

In [ ]:
from openpyxl import load_workbook
ma = load_workbook(BASE / "05_reto_gimnasios/02_BASE_12_CCAA_INTEGRADAS.xlsx", data_only=True)["Madrid"]["B7"].value
cv = load_workbook(BASE / "05_reto_gimnasios/CCAA_PENDIENTES/COMUNITAT_VALENCIANA.xlsx", data_only=True)["C. Valenciana"]["B7"].value
print("MADRID               B7 =", repr(ma), "→", type(ma).__name__)
print("COMUNITAT VALENCIANA B7 =", repr(cv), "→", type(cv).__name__)

In [ ]:
#@title Ver la respuesta (ejecútala cuando hayas pensado la tuya)
revelar("""
**LO QUE LEE EL PROGRAMA:** `12` → número (`int`) · `'12'` → texto (`str`).
**El aspecto no determina el tipo de dato.** No bases el diagnóstico en la alineación, en los triángulos verdes ni en el formato de celda: dependen de la configuración y pueden engañar. La prueba es lo que lee el programa.
""")

## PROMPT 2 · Corregimos la causa identificada

Ahora sí conocemos la causa. Aporta la evidencia y pide un cambio **limitado** (PROMPT 2):

```
He comprobado que los recuentos de Comunitat Valenciana están almacenados como texto aunque contienen valores numéricos.

Modifica únicamente la lectura de esos valores para:
- eliminar espacios normales o no separables;
- intentar convertir cadenas numéricas a enteros;
- registrar una incidencia cuando un valor no pueda convertirse;
- no sustituir silenciosamente valores desconocidos por 0.

Mantén el resto del programa sin cambios.
```

*Cambio concreto, evidencia concreta. Diagnosticar primero evita reescribir todo el programa.*
Si tu herramienta ya daba *25 / 25*, revisa si tu conversión registra incidencias o convierte en 0 en silencio.

### Conversión segura

**limpiar → intentar convertir → si convierte: usar → si no convierte: registrar incidencia.**
*Nunca sustituir silenciosamente lo desconocido por 0.* **0 es un dato real** (cero controles); **no convertible** es otra situación: hay que verla.

Pega aquí la **nueva versión** de tu herramienta (sustituye a la anterior) y vuelve a ejecutarla:

In [ ]:
ir_a("05_reto_gimnasios")

In [ ]:
%%writefile mi_herramienta.py
# ↓ Borra estas dos líneas y pega aquí, completo, el código que te ha dado tu IA ↓

In [ ]:
!python mi_herramienta.py

## Volvemos a ejecutar

**Antes de mirar, predice:** ¿qué debería cambiar y qué debería quedarse igual? (Es el *cambio controlado* de CONTROLES, ahora con la herramienta completa.)

Compara el `CONTROL_CARGA` **ANTES** (la primera base) con el **DESPUÉS** (tu libro corregido), fila de Comunitat Valenciana:

In [ ]:
print("ANTES · CONSOLIDADO_BASE.xlsx · CONTROL_CARGA")
ver_excel(BASE / "05_reto_gimnasios/SALIDAS/CONSOLIDADO_BASE.xlsx", "CONTROL_CARGA", buscar="Valenciana", columnas=9)
print()
print("DESPUÉS · tu libro corregido · CONTROL_CARGA")
ver_excel(MI_SALIDA, "CONTROL_CARGA", buscar="Valenciana", columnas=9)

In [ ]:
#@title Ver la respuesta (ejecútala cuando hayas pensado la tuya)
revelar("""
| CONTROL | ANTES | DESPUÉS | |
|---|:-:|:-:|:-:|
| CCAA | 17 | 17 | igual |
| Registros | 425 | 425 | igual |
| CV · preguntas detectadas | 25 | 25 | igual |
| CV · valores válidos | 0 | **25** | cambia |
| Controles nacionales por pregunta | 327 | **351** | cambia |

**DEBÍA PERMANECER IGUAL:** 17 CCAA · 425 registros · 25 preguntas detectadas. **DEBÍA CAMBIAR:** CV válidas 0 → 25; nacional 327 → 351.
*La corrección ha cambiado exactamente lo que esperábamos. Y no ha alterado lo que debía permanecer igual.*
""")

## Un solo «OK» no basta

✓ El programa ha terminado sin errores · ✓ 17 CCAA · 425 registros · ✗ …y aun así Comunitat Valenciana no participaba en el cálculo.
Cada control responde a una pregunta distinta. Ordenémoslos en cinco preguntas.

## Cinco preguntas antes de confiar

### Capa 1 · Cobertura — ¿Está todo?
17 CCAA · 425 registros · 25 preguntas por CCAA. Esto estaba bien incluso cuando Comunitat Valenciana no participaba correctamente en los cálculos. **Cobertura correcta ≠ resultado correcto.**

### Capa 2 · Tipo / interpretación — ¿He entendido correctamente cada dato?
`12` número frente a `'12'` texto (Madrid frente a Comunitat Valenciana, pregunta 1): ésa fue la causa. En VARIACIONES, CV parecía estructuralmente normal y `CONTROL_NORMALIZACION` dio OK: era correcto, pero ese OK respondía a otra pregunta. **Detectar un dato no significa haberlo interpretado bien.**

### Capa 3 · Coherencia — ¿Se cumplen las relaciones internas?
Madrid · pregunta 1: Sí 12 · No 15 · No procede 0 → 12 + 15 + 0 = 27 controles. La misma relación que conservamos en la muestra sintética y comprobamos en cada fila del consolidado. **Los datos deben respetar las relaciones que conocemos.**

### Capa 4 · Reconciliación — ¿El resultado coincide con un control independiente?
Caso pequeño · 2 CCAA: 27 + 22 = 49 por pregunta (comparado con `RESULTADO_ESPERADO_2_CCAA.xlsx`). Campaña estatal: 351 controles por pregunta (327 en la primera base: faltaba una comunidad en el cálculo). **Un total independiente puede detectar errores que una comprobación estructural no ve.**

### Capa 5 · Lógica — ¿Estoy calculando realmente lo que quería calcular?
52,5 % (promediar porcentajes) frente a 36 % (sumar recuentos, después calcular): lo vimos en REGLAS con un ejemplo simplificado. ¿Podría pasar en la herramienta real? **Un programa puede ejecutar perfectamente una regla equivocada.**

## Micro-auditoría · ambos pasan los primeros controles

| | Informe A | Informe B |
|---|:-:|:-:|
| Cobertura | 17 CCAA · 425 registros · 425 válidos · 351 controles | 17 CCAA · 425 registros · 425 válidos · 351 controles |
| P15 · recuentos | Sí 154 · No 23 · NP 174 | Sí 154 · No 23 · NP 174 |
| P15 · Sí · No · NP | **44,7 % · 6,2 % · 49,1 %** | **43,9 % · 6,6 % · 49,6 %** |

Los dos pasan las cuatro primeras capas. **¿Cuál firmarías? ¿Qué control falta?** *(Pista: ¿cuánto es 154 entre 351?)*
> 🟢 **IGUAL** — micro-auditoría y cinco capas, como en clase (en papel).

In [ ]:
#@title Ver la respuesta (ejecútala cuando hayas pensado la tuya)
revelar("""
**INFORME A · INCORRECTO — promedia porcentajes autonómicos:** (%Sí CCAA 1 + … + %Sí CCAA 17) / 17 = 44,7 % (No 6,2 % · No procede 49,1 %).
↳ da el mismo peso a una comunidad con 12 controles que a una con 34.

**INFORME B · CORRECTO — suma recuentos y después calcula:** (Σ Sí) / (Σ controles) = 154 / 351 = **43,9 %** (No 23/351 = 6,6 % · No procede 174/351 = 49,6 %).
↳ cada control cuenta lo mismo, venga de la comunidad que venga.

*Un resultado puede superar cobertura, tipos, coherencia y reconciliación y seguir siendo incorrecto por la lógica de cálculo.*
""")

In [ ]:
print("154 / 351 =", round(154 / 351 * 100, 1), "%")

## Cinco preguntas antes de confiar (resumen)

| | | Ejemplo vivido |
|---|---|---|
| 1 | **COBERTURA** · ¿Está todo? | 17 CCAA · 425 registros |
| 2 | **TIPO / INTERPRETACIÓN** · ¿He entendido correctamente cada dato? | 12 frente a '12' |
| 3 | **COHERENCIA** · ¿Se cumplen las relaciones internas? | Sí + No + No procede = controles |
| 4 | **RECONCILIACIÓN** · ¿El resultado coincide con un control independiente? | 49 en el caso pequeño · 351 nacional |
| 5 | **LÓGICA** · ¿Estoy calculando realmente lo que quería calcular? | 52,5 % frente a 36 % · informe A |

*Ninguna capa sustituye a las demás.*

## PROMPT 3 · Construye un auditor independiente

Si estas cinco preguntas son importantes, conviene que las responda un programa. Pero no la propia herramienta diciéndose que está bien: **un segundo programa**.
Este es el PROMPT 3, con una última línea para Colab:

```
Escribe un programa independiente que reciba como entrada el Excel generado por nuestra herramienta y lo audite sin modificarlo.

Debe comprobar cinco capas:
1. COBERTURA: 17 CCAA; 25 preguntas por comunidad; 425 registros.
2. TIPO / INTERPRETACIÓN: los recuentos y totales necesarios son numéricos; identifica los valores no interpretables.
3. COHERENCIA: Sí + No + No procede = controles realizados.
4. RECONCILIACIÓN: existen 25 preguntas nacionales y el total por pregunta coincide con el control independiente esperado —351 en estos materiales—.
5. LÓGICA: recalcula los porcentajes nacionales a partir de los recuentos agregados y comprueba que coinciden con los publicados; no aceptes como correcto un promedio simple de porcentajes autonómicos.

Genera un informe con OK / REVISAR / NO EVALUABLE para cada capa.

No modifiques el archivo auditado.

Lo ejecutaré en Google Colab con "!python mi_auditar_salida.py --entrada <ruta del Excel>" desde la carpeta 06_diagnostico_y_validacion.
```

*Un segundo programa revisa la salida del primero.* (`NO EVALUABLE` existe para no confundir «no he podido comprobarlo» con «está bien».)

*En un caso real, el 351 debe proceder de una fuente, cálculo o referencia independiente del programa cuya salida estamos auditando: aquí es el control fijado para el ejercicio.*

## Mini-práctica · Ejecuta el auditor sobre tu salida

**1** obtener el auditor (con el prompt o la referencia) → **2** ejecutarlo sobre el Excel de tu herramienta → **3** leer el informe capa a capa → **4** si alguna capa dice REVISAR o NO EVALUABLE: ¿qué pregunta no ha superado?
*El auditor no corrige nada: informa.*

> 🟡 **ADAPTADA** — guardas el script con `%%writefile` y lo ejecutas con `!python`.

In [ ]:
ir_a("06_diagnostico_y_validacion")

In [ ]:
%%writefile mi_auditar_salida.py
# ↓ Borra estas dos líneas y pega aquí, completo, el código que te ha dado tu IA ↓

In [ ]:
!python mi_auditar_salida.py --entrada ../05_reto_gimnasios/SALIDAS/MI_RESULTADO_17_CCAA.xlsx

## Resultado real de la auditoría · demo de referencia

En clase el docente ejecuta `auditar_salida.py` (la referencia) sobre la **solución corregida**: *un programa independiente revisa la salida del primero.*
La celda siguiente ejecuta esa referencia sobre la solución de referencia (si no existe, la genera antes con `02_SOLUCION_COMPLETA.py`).
Ejecútala **después** de haber probado tu propio auditor.

> 🟡 **ADAPTADA** — en clase lo proyecta el docente; aquí lo ejecutas tú. Genera `SOLUCION_COMPLETA_17_CCAA.xlsx` como cualquier otra referencia.

In [ ]:
ir_a("06_diagnostico_y_validacion")
SOL = BASE / "05_reto_gimnasios/SALIDAS/SOLUCION_COMPLETA_17_CCAA.xlsx"
if not SOL.exists():
    !python ../99_ayudas_y_referencias/02_SOLUCION_COMPLETA.py
!python auditar_salida.py

## Un auditor útil también debe saber decir «REVISAR»

*¿Y si la salida estuviera mal? ¿Lo detectaría?* Comprobamos el comprobador: creamos el **informe A** (la misma solución, con los
porcentajes nacionales sustituidos por el promedio simple de los porcentajes autonómicos) y le pasamos el auditor.

> 🟡 **ADAPTADA** — en clase este caso lo construye el docente con una herramienta suya (no incluida en el material del alumnado); aquí lo reproduce esta celda con unas líneas de código.

In [ ]:
# Construye el "informe A": copia de la solución con los % nacionales = promedio simple de los % autonómicos (lógica errónea)
from collections import defaultdict
from openpyxl import load_workbook

SOL = BASE / "05_reto_gimnasios/SALIDAS/SOLUCION_COMPLETA_17_CCAA.xlsx"
if not SOL.exists():
    ir_a("99_ayudas_y_referencias")
    !python 02_SOLUCION_COMPLETA.py
INFORME_A = BASE / "05_reto_gimnasios/SALIDAS/INFORME_A_PROMEDIO_DE_PORCENTAJES.xlsx"
shutil.copy(SOL, INFORME_A)
wb = load_workbook(INFORME_A)
d = wb["DATOS_CONSOLIDADOS"]
h = {c.value: c.column - 1 for c in d[1]}
pc = defaultdict(list)
for r in d.iter_rows(min_row=2, values_only=True):
    if r[0] is None:
        continue
    t = r[h["realizados"]]
    pc[r[h["numero_pregunta"]]].append((r[h["si"]] / t, r[h["no"]] / t, r[h["no_procede"]] / t))
nac = wb["NACIONAL"]
cab = {}
for c in nac[2]:
    cab.setdefault(c.value, c.column)
for fila in range(3, 28):
    n = nac.cell(fila, cab["numero_pregunta"]).value
    for j, k in enumerate(("% si", "% no", "% no procede")):
        nac.cell(fila, cab[k]).value = sum(x[j] for x in pc[n]) / len(pc[n])
wb.save(INFORME_A)
print("Creado:", INFORME_A.name)

In [ ]:
ir_a("06_diagnostico_y_validacion")
!python auditar_salida.py --entrada ../05_reto_gimnasios/SALIDAS/INFORME_A_PROMEDIO_DE_PORCENTAJES.xlsx

*El auditor no siempre devuelve OK. También comprobamos el comprobador.* (Debe dar 1–4 OK y **5 LÓGICA = REVISAR**.)

## Dos capas de defensa

1. **AUDITOR INDEPENDIENTE** — un segundo programa revisa la salida, sin modificarla (`auditar_salida.py`) = **hilo principal**.
2. **CONTROLES INTEGRADOS** — la propia herramienta documenta y comprueba mientras trabaja (`CONTROL_CARGA` ampliado + hoja `COMPROBACIONES`) = **ampliación · endurecimiento**.

No son excluyentes. El hilo principal termina con el auditor; los controles integrados son una ampliación y **no sustituyen al auditor**.

---
# Ampliación opcional · la herramienta también debe explicar qué ha comprobado

*(Diapositivas 108–109)* · No es necesario que todos la hagan.

## PROMPT 4 · Reforzar los controles integrados

```
Sin cambiar las reglas de cálculo, amplía los controles de la herramienta para comprobar y registrar como mínimo:
- CCAA esperadas y encontradas;
- preguntas esperadas y encontradas;
- valores numéricos válidos;
- incidencias de conversión;
- coherencia Sí + No + No procede = realizados;
- totales por pregunta;
- diferencias respecto de controles independientes conocidos;
- lógica de cálculo: recalcula los porcentajes nacionales a partir de los recuentos agregados y comprueba que coinciden con los publicados (no un promedio simple de porcentajes autonómicos).

No ocultes errores ni los conviertas en cero. Si una comprobación no puede realizarse, indícalo expresamente.
```

*Los controles también se programan.* **Objetivo:** que `CONTROL_CARGA` no sea solo un registro de lectura, sino una herramienta de diagnóstico.

Añade al final de tu mensaje: *«Lo ejecutaré con !python mi_herramienta_v2.py desde la carpeta 05_reto_gimnasios y guardaré el resultado en SALIDAS/MI_RESULTADO_17_CCAA_V2.xlsx.»*
Pega el código aquí y ejecútalo (`CONTROL_CARGA` ampliado y hoja nueva `COMPROBACIONES`: una fila por comprobación y capa):

In [ ]:
ir_a("05_reto_gimnasios")

In [ ]:
%%writefile mi_herramienta_v2.py
# ↓ Borra estas dos líneas y pega aquí, completo, el código que te ha dado tu IA ↓

In [ ]:
!python mi_herramienta_v2.py

In [ ]:
V2 = BASE / "05_reto_gimnasios/SALIDAS/MI_RESULTADO_17_CCAA_V2.xlsx"
hojas(V2)
ver_excel(V2, "COMPROBACIONES", filas=(1, 12), columnas=5)

*Si el tiempo aprieta: la referencia `99_ayudas_y_referencias/03_SOLUCION_CONTROLES_REFORZADOS.py` está en «Solo si te has bloqueado».*

## Otra ampliación (material, no vista en la sesión) · cambio controlado con la herramienta completa

`07_ampliacion_cambio_controlado/ARAGON_CORREGIDO.xlsx` cambia únicamente la pregunta 1 de Aragón:

- antes: Sí 8 · No 10 · No procede 0
- después: Sí 9 · No 9 · No procede 0
- total: 18 → 18

**Antes de ejecutar, predice** qué debe cambiar y qué debe permanecer igual. Trabaja sobre una copia y compara después con tu predicción.
Si aparecen a la vez `ARAGON.xlsx` y `ARAGON_CORREGIDO.xlsx`, una herramienta robusta debería señalar la ambigüedad en vez de elegir o sumar silenciosamente ambos.

In [ ]:
ver_excel(BASE / "07_ampliacion_cambio_controlado/ARAGON_CORREGIDO.xlsx", "Aragón", filas=(6, 8), columnas=5)

---
# Cierre

## La herramienta ha crecido. Los controles también.

Las ocho piezas siguen en verde; CONTROLES queda **profundizado**: casos conocidos · cambio controlado · auditoría en cinco capas.

- **AL PRINCIPIO:** «¿Ha terminado?»
- **AHORA:** «¿Está todo? ¿Lo he interpretado bien? ¿Cuadra? ¿Se reconcilia? ¿La lógica es correcta?»

## Una automatización fiable permite demostrar por qué confiamos

**OBJETIVO → DIAGNÓSTICO → INFO. PARA LA IA → SOLUCIÓN → COMPROBACIÓN**

Una automatización fiable no solo produce resultados. Expone qué ha utilizado, qué incidencias ha encontrado y qué controles ha superado.

comprobar archivos → entender estructuras → consolidar datos → detectar fallos silenciosos → auditar la lógica

✅ **Bloque 2 completado.**

---
# Solo si te has bloqueado

Referencias del material, **en el orden del bloque**. Úsalas cuando te quedes sin salida y quieras continuar desde la comprobación.
Recuerda: *usar una referencia forma parte del ejercicio. Comprobarla también.*

⚠️ Algunas referencias escriben archivos con el mismo nombre que podrían haber creado tus programas
(por ejemplo `SALIDAS_NORMALIZADAS/…_NORMALIZADO.xlsx`): **sobrescribirán** los tuyos.

| Práctica | Referencia |
|---|---|
| Reto · documentación | `01_reto_documentacion/AYUDAS_Y_REFERENCIAS/solucion_nivel1.py` |
| Muestra sintética | `02_muestra_sintetica/generar_muestra_sintetica.py` |
| Combinar dos CCAA + cambio controlado | `03_modelo_2_ccaa/CONTROLES/combinar_2_ccaa.py` |
| Normalizar + comprobar | `05_reto_gimnasios/NORMALIZACION/normalizar_entradas.py` y `comprobar_normalizacion.py` |
| Reto campaña estatal (versión corregida) | `99_ayudas_y_referencias/02_SOLUCION_COMPLETA.py` |
| Diagnóstico de tipos | `06_diagnostico_y_validacion/diagnosticar_tipos.py` |
| Auditor independiente | `06_diagnostico_y_validacion/auditar_salida.py` |
| Controles integrados (ampliación) | `99_ayudas_y_referencias/03_SOLUCION_CONTROLES_REFORZADOS.py` |

Todas se ejecutan con `!python` desde su carpeta. Para **leer** una referencia: `!cat nombre.py`.
Los prompts de apoyo están en `99_ayudas_y_referencias/prompts_bloque_2.md` y `AYUDAS_Y_REFERENCIAS/PROMPTS_RETO_DOCUMENTACION.txt`.

### Reto · documentación

Genera `SALIDAS/resultado_comprobacion.xlsx` (resultado de referencia: **21 / 3 / 1 / 2**).

> 🔴 **NO EXPORTABLE** — `solucion_nivel2.py` / `.pyw` (la interfaz con `tkinter`): no abre ventanas en Colab. Puedes leer su código con `!cat solucion_nivel2.py`, pero no ejecutarlo.

In [ ]:
ir_a("01_reto_documentacion/AYUDAS_Y_REFERENCIAS")
!python solucion_nivel1.py
ver_excel("../SALIDAS/resultado_comprobacion.xlsx", filas=(1, 8), columnas=7)

### Muestra sintética

Genera `MUESTRA_MADRID_PARA_IA.xlsx` (semilla 42; hojas `MUESTRA_MADRID` y `COMPROBACION`). La copia ya generada está en `REFERENCIA/`.

In [ ]:
ir_a("02_muestra_sintetica")
!python generar_muestra_sintetica.py
ver_excel("MUESTRA_MADRID_PARA_IA.xlsx", "MUESTRA_MADRID", filas=(5, 9), columnas=4)

### Combinar dos CCAA y cambio controlado

Genera `SALIDAS/CONSOLIDADO_2_CCAA_GENERADO.xlsx` y, con el Madrid modificado, `CAMBIO_CONTROLADO/SALIDA_CAMBIO.xlsx`.

In [ ]:
ir_a("03_modelo_2_ccaa/CONTROLES")
!python combinar_2_ccaa.py
!python combinar_2_ccaa.py --madrid CAMBIO_CONTROLADO/MADRID_CAMBIO_CONTROLADO.xlsx --salida CAMBIO_CONTROLADO/SALIDA_CAMBIO.xlsx
ver_excel("SALIDAS/CONSOLIDADO_2_CCAA_GENERADO.xlsx", "CONSOLIDADO_2_CCAA", filas=(1, 4), columnas=6)
ver_excel("CAMBIO_CONTROLADO/SALIDA_CAMBIO.xlsx", "CONSOLIDADO_2_CCAA", filas=(1, 4), columnas=6)

### Normalizar y comprobar

Genera los 5 `*_NORMALIZADO.xlsx` (hoja `DATOS`) y `CONTROL_NORMALIZACION.xlsx`. Copia ya generada en `NORMALIZACION/REFERENCIA/`.

In [ ]:
ir_a("05_reto_gimnasios/NORMALIZACION")
!python normalizar_entradas.py
!python comprobar_normalizacion.py
ver_excel("SALIDAS_NORMALIZADAS/CONTROL_NORMALIZACION.xlsx", filas=(1, 8), columnas=12)

### Reto campaña estatal · solución corregida (lectura segura de números almacenados como texto)

Genera `05_reto_gimnasios/SALIDAS/SOLUCION_COMPLETA_17_CCAA.xlsx` (copia de referencia: `99_ayudas_y_referencias/04_SOLUCION_REFERENCIA_17_CCAA.xlsx`).

In [ ]:
ir_a("99_ayudas_y_referencias")
!python 02_SOLUCION_COMPLETA.py

In [ ]:
ver_excel(BASE / "05_reto_gimnasios/SALIDAS/SOLUCION_COMPLETA_17_CCAA.xlsx", "CONTROL_CARGA", buscar="Valenciana", columnas=9)
graficos(BASE / "05_reto_gimnasios/SALIDAS/SOLUCION_COMPLETA_17_CCAA.xlsx")
# descargar(BASE / "05_reto_gimnasios/SALIDAS/SOLUCION_COMPLETA_17_CCAA.xlsx")   # ← para ver las gráficas dibujadas

### Diagnóstico de tipos y auditor independiente

`diagnosticar_tipos.py` solo observa tipos (no modifica archivos) y escribe `SALIDAS/diagnostico_tipos.csv`. `auditar_salida.py` escribe `SALIDAS/AUDITORIA_SALIDA.txt`; sin `--entrada` audita la solución completa.

In [ ]:
ir_a("06_diagnostico_y_validacion")
!python diagnosticar_tipos.py

In [ ]:
ir_a("06_diagnostico_y_validacion")
!python auditar_salida.py
# Sobre otro archivo:  !python auditar_salida.py --entrada ../05_reto_gimnasios/SALIDAS/MI_RESULTADO_17_CCAA.xlsx

### Controles integrados (ampliación)

Genera `SALIDAS/SOLUCION_CONTROLES_REFORZADOS.xlsx` con `CONTROL_CARGA` ampliado y la hoja `COMPROBACIONES` (copia de referencia: `99_ayudas_y_referencias/05_SOLUCION_CONTROLES_REFORZADOS_REFERENCIA.xlsx`).

In [ ]:
ir_a("99_ayudas_y_referencias")
!python 03_SOLUCION_CONTROLES_REFORZADOS.py
ver_excel(BASE / "05_reto_gimnasios/SALIDAS/SOLUCION_CONTROLES_REFORZADOS.xlsx", "COMPROBACIONES", filas=(1, 9), columnas=5)

---
## Restaurar los materiales originales

Por ejemplo, si has estropeado un archivo del material. **Atención:** borra también todo lo que hayas creado
(`mi_*.py`, salidas, `MI_*.xlsx`…) y los resultados de las referencias.

In [ ]:
import os, zipfile, shutil, urllib.request
from pathlib import Path

ZIP_URL = "https://raw.githubusercontent.com/consumer-spain/curso-ia-colab/main/bloque2/BLOQUE2_MATERIALES.zip"
NOMBRE_BASE = "BLOQUE2_MATERIALES_ACTUALIZADOS"
_aqui = Path.cwd().resolve()
TRABAJO = Path(*_aqui.parts[:_aqui.parts.index(NOMBRE_BASE)]) if NOMBRE_BASE in _aqui.parts else _aqui
BASE = TRABAJO / NOMBRE_BASE
ZIP_LOCAL = TRABAJO / "materiales_b2.zip"

os.chdir(TRABAJO)                       # salimos de la carpeta antes de borrarla
shutil.rmtree(BASE, ignore_errors=True)
if not ZIP_LOCAL.exists():
    urllib.request.urlretrieve(ZIP_URL, ZIP_LOCAL)
with zipfile.ZipFile(ZIP_LOCAL) as z:
    z.extractall(TRABAJO)
print("Materiales restaurados en:", BASE)
if "ir_a" in globals():
    ir_a("")
else:
    print("Ejecuta ahora la celda Preparación para volver a cargar las herramientas.")